In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

print("Catalog:", spark.catalog.currentCatalog())
print("Schema:", spark.catalog.currentDatabase())

Catalog: workspace
Schema: default


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA silver_layer")

print("Catalog:", spark.catalog.currentCatalog())
print("Schema:", spark.catalog.currentDatabase())

Catalog: workspace
Schema: silver_layer


In [0]:
candidate_tables = [
    "silver_stations_candidate",
    "silver_chargers_candidate",
    "silver_sessions_candidate",
    "silver_maintenance_candidate"
]

for table in candidate_tables:
    print(f"\n===== {table} =====")
    
    if spark.catalog.tableExists(table):
        print("EXISTS")
        print("Rows:", spark.table(table).count())
        spark.table(table).printSchema()
    else:
        print("MISSING")


===== silver_stations_candidate =====
EXISTS
Rows: 180
root
 |-- physical_record_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- city_band: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- operator_code: string (nullable = true)
 |-- connector_capacity: integer (nullable = true)
 |-- operating_start_hour: integer (nullable = true)
 |-- operating_end_hour: integer (nullable = true)
 |-- is_24x7: boolean (nullable = true)
 |-- commission_date: date (nullable = true)
 |-- station_status: string (nullable = true)
 |-- latitude_band: double (nullable = true)
 |-- longitude_band: double (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: 

In [0]:
stations = spark.table("silver_stations_candidate")

stations.printSchema()
stations.show(5, truncate=False)

root
 |-- physical_record_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- city_band: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- operator_code: string (nullable = true)
 |-- connector_capacity: integer (nullable = true)
 |-- operating_start_hour: integer (nullable = true)
 |-- operating_end_hour: integer (nullable = true)
 |-- is_24x7: boolean (nullable = true)
 |-- commission_date: date (nullable = true)
 |-- station_status: string (nullable = true)
 |-- latitude_band: double (nullable = true)
 |-- longitude_band: double (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)

+------------------+----------

In [0]:
chargers = spark.table("silver_chargers_candidate")

chargers.printSchema()
chargers.show(5, truncate=False)

root
 |-- charger_id: string (nullable = true)
 |-- charger_label: string (nullable = true)
 |-- connector_position: long (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- firmware_major: long (nullable = true)
 |-- install_date: string (nullable = true)
 |-- manufacturer_band: string (nullable = true)
 |-- operational_status: string (nullable = true)
 |-- physical_record_id: string (nullable = true)
 |-- rated_power_kw: double (nullable = true)
 |-- smart_meter_enabled: boolean (nullable = true)
 |-- source_system: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)

+----------+-------------+------------------+--------------+--------------+------------+-----------------+------------------+------------------+----

In [0]:
sessions = spark.table("silver_sessions_candidate")

sessions.printSchema()
sessions.show(5, truncate=False)

root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

In [0]:
maintenance = spark.table("silver_maintenance_candidate")

maintenance.printSchema()
maintenance.show(5, truncate=False)

root
 |-- physical_record_id: string (nullable = true)
 |-- maintenance_id: string (nullable = true)
 |-- incident_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- fault_category: string (nullable = true)
 |-- fault_code: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- status_after: string (nullable = true)
 |-- related_fault_event_id: string (nullable = true)
 |-- planned_flag: boolean (nullable = true)
 |-- notes_code: string (nullable = true)
 |-- batch_date: date (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)

+------------------+--------------+----------

In [0]:
from pyspark.sql import functions as F, Window
from functools import reduce
from datetime import datetime, timezone

dq_run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
dq_checked_ts = datetime.now(timezone.utc).isoformat()

print("dq_run_id =", dq_run_id)
print("dq_checked_ts =", dq_checked_ts)

dq_run_id = 20260910T085320Z
dq_checked_ts = 2026-09-10T08:53:20.566494+00:00


In [0]:
stations = spark.table("silver_stations_candidate")

print("Station candidate count:", stations.count())

stations.select(
    "station_id",
    "connector_capacity",
    "operating_start_hour",
    "operating_end_hour",
    "is_24x7",
    "commission_date",
    "site_type",
    "station_status"
).summary().show(truncate=False)

Station candidate count: 180
+-------+----------+------------------+--------------------+------------------+---------------+---------------+
|summary|station_id|connector_capacity|operating_start_hour|operating_end_hour|site_type      |station_status |
+-------+----------+------------------+--------------------+------------------+---------------+---------------+
|count  |179       |180               |180                 |180               |180            |180            |
|mean   |NULL      |6.627777777777778 |2.816666666666667   |23.083333333333332|NULL           |NULL           |
|stddev |NULL      |1.596191541255285 |3.5696971062217004  |1.6908395983907027|NULL           |NULL           |
|min    |STN0001   |0                 |0                   |6                 |HIGHWAY_STOP   |ACTIVE         |
|25%    |NULL      |5                 |0                   |22                |NULL           |NULL           |
|50%    |NULL      |7                 |0                   |24             

In [0]:
stations.select(
    "site_type",
    "station_status",
    "is_24x7"
).groupBy(
    "site_type",
    "station_status",
    "is_24x7"
).count().orderBy(
    "site_type",
    "station_status"
).show(100, truncate=False)

+-------------------+---------------+-------+-----+
|site_type          |station_status |is_24x7|count|
+-------------------+---------------+-------+-----+
|HIGHWAY_STOP       |ACTIVE         |true   |25   |
|HIGHWAY_STOP       |LIMITED_SERVICE|true   |1    |
|MALL               |ACTIVE         |true   |6    |
|MALL               |ACTIVE         |false  |19   |
|METRO_HUB          |ACTIVE         |false  |16   |
|METRO_HUB          |ACTIVE         |true   |9    |
|OFFICE_PARK        |ACTIVE         |true   |7    |
|OFFICE_PARK        |ACTIVE         |false  |18   |
|OFFICE_PARK        |LIMITED_SERVICE|false  |1    |
|PUBLIC_PARKING     |ACTIVE         |true   |50   |
|PUBLIC_PARKING     |LIMITED_SERVICE|true   |1    |
|RESIDENTIAL_CLUSTER|ACTIVE         |true   |4    |
|RESIDENTIAL_CLUSTER|ACTIVE         |false  |21   |
|RESIDENTIAL_CLUSTER|LIMITED_SERVICE|true   |1    |
|ROOFTOP_UNKNOWN    |LIMITED_SERVICE|false  |1    |
+-------------------+---------------+-------+-----+



In [0]:
station_checks = (
    stations
    .withColumn(
        "DQ_CHG_002_check",
        F.when(
            # Station key
            F.col("station_id").isNull() |
            (F.trim(F.col("station_id")) == "") |

            # Capacity
            F.col("connector_capacity").isNull() |
            (F.col("connector_capacity") <= 0) |

            # Operating hours
            F.col("operating_start_hour").isNull() |
            F.col("operating_end_hour").isNull() |
            (F.col("operating_start_hour") < 0) |
            (F.col("operating_start_hour") > 23) |
            (F.col("operating_end_hour") < 0) |
            (F.col("operating_end_hour") > 24) |

            # 24x7 consistency
            (
                F.col("is_24x7").isNull() |
                (
                    F.col("is_24x7") &
                    (
                        (F.col("operating_start_hour") != 0) |
                        (F.col("operating_end_hour") != 24)
                    )
                )
            ) |

            # Commissioning date
            F.col("commission_date").isNull() |

            # Site type
            F.col("site_type").isNull() |
            (F.trim(F.col("site_type")) == "") |

            # Station status
            F.col("station_status").isNull() |
            (F.trim(F.col("station_status")) == ""),

            F.lit("FAIL")
        ).otherwise(F.lit("PASS"))
    )
)

station_checks.select(
    "physical_record_id",
    "station_id",
    "connector_capacity",
    "operating_start_hour",
    "operating_end_hour",
    "is_24x7",
    "commission_date",
    "site_type",
    "station_status",
    "DQ_CHG_002_check"
).filter(
    F.col("DQ_CHG_002_check") == "FAIL"
).show(50, truncate=False)

+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+----------------+
|physical_record_id|station_id|connector_capacity|operating_start_hour|operating_end_hour|is_24x7|commission_date|site_type          |station_status |DQ_CHG_002_check|
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+----------------+
|STNREC000176      |NULL      |0                 |0                   |24                |true   |2022-09-27     |PUBLIC_PARKING     |LIMITED_SERVICE|FAIL            |
|STNREC000177      |STN0177   |6                 |23                  |6                 |true   |2023-02-09     |RESIDENTIAL_CLUSTER|LIMITED_SERVICE|FAIL            |
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+----------

In [0]:
station_checks.groupBy("DQ_CHG_002_check").count().show()

+----------------+-----+
|DQ_CHG_002_check|count|
+----------------+-----+
|            PASS|  178|
|            FAIL|    2|
+----------------+-----+



In [0]:
station_checks.filter(
    F.col("DQ_CHG_002_check") == "FAIL"
).select(
    "physical_record_id",
    "station_id",
    "connector_capacity",
    "operating_start_hour",
    "operating_end_hour",
    "is_24x7",
    "commission_date",
    "site_type",
    "station_status"
).show(truncate=False)

+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+
|physical_record_id|station_id|connector_capacity|operating_start_hour|operating_end_hour|is_24x7|commission_date|site_type          |station_status |
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+
|STNREC000176      |NULL      |0                 |0                   |24                |true   |2022-09-27     |PUBLIC_PARKING     |LIMITED_SERVICE|
|STNREC000177      |STN0177   |6                 |23                  |6                 |true   |2023-02-09     |RESIDENTIAL_CLUSTER|LIMITED_SERVICE|
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+



In [0]:
station_checks.filter(
    F.col("DQ_CHG_002_check") == "FAIL"
).select(
    "physical_record_id",
    "station_id",
    "connector_capacity",
    "operating_start_hour",
    "operating_end_hour",
    "is_24x7",
    "commission_date",
    "site_type",
    "station_status"
).show(20, truncate=False)

+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+
|physical_record_id|station_id|connector_capacity|operating_start_hour|operating_end_hour|is_24x7|commission_date|site_type          |station_status |
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+
|STNREC000176      |NULL      |0                 |0                   |24                |true   |2022-09-27     |PUBLIC_PARKING     |LIMITED_SERVICE|
|STNREC000177      |STN0177   |6                 |23                  |6                 |true   |2023-02-09     |RESIDENTIAL_CLUSTER|LIMITED_SERVICE|
+------------------+----------+------------------+--------------------+------------------+-------+---------------+-------------------+---------------+



In [0]:
from pyspark.sql.functions import (
    col, lit, when, concat_ws, current_timestamp
)

# ---------------------------------------
# DQ-CHG-002 routing for Stations
# ---------------------------------------

station_routed = (
    station_checks
    .withColumn(
        "failure_count",
        when(col("DQ_CHG_002_check") == "FAIL", lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "failed_rule_ids",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("DQ-CHG-002")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "failure_reasons",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            concat_ws(
                "; ",
                when(
                    col("station_id").isNull() | (col("station_id") == ""),
                    lit("station_id is null or blank")
                ),
                when(
                    col("connector_capacity").isNull()
                    | (col("connector_capacity") <= 0),
                    lit("connector_capacity is null or <= 0")
                ),
                when(
                    col("operating_start_hour").isNull()
                    | (col("operating_start_hour") < 0)
                    | (col("operating_start_hour") > 23),
                    lit("operating_start_hour is invalid")
                ),
                when(
                    col("operating_end_hour").isNull()
                    | (col("operating_end_hour") < 0)
                    | (col("operating_end_hour") > 24),
                    lit("operating_end_hour is invalid")
                ),
                when(
                    col("is_24x7")
                    & (
                        (col("operating_start_hour") != 0)
                        | (col("operating_end_hour") != 24)
                    ),
                    lit("24x7 station has inconsistent operating hours")
                ),
                when(
                    col("commission_date").isNull(),
                    lit("commission_date is null")
                ),
                when(
                    col("site_type").isNull() | (col("site_type") == ""),
                    lit("site_type is null or blank")
                ),
                when(
                    col("station_status").isNull() | (col("station_status") == ""),
                    lit("station_status is null or blank")
                )
            )
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "highest_severity",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("CRITICAL")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "dq_status",
        when(col("DQ_CHG_002_check") == "FAIL", lit("QUARANTINE"))
        .otherwise(lit("TRUSTED"))
    )
    .withColumn("dq_run_id", lit(dq_run_id))
    .withColumn("dq_checked_ts", current_timestamp())
    .withColumn(
        "rule_id",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("DQ-CHG-002")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "rule_name",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("Station master integrity")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "severity",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("CRITICAL")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "rework_status",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            lit("PENDING")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "quarantined_at",
        when(
            col("DQ_CHG_002_check") == "FAIL",
            current_timestamp()
        ).otherwise(lit(None).cast("timestamp"))
    )
)

# ---------------------------------------
# Trusted stations
# ---------------------------------------

trusted_stations = (
    station_routed
    .filter(col("DQ_CHG_002_check") == "PASS")
    .drop("DQ_CHG_002_check")
)

# ---------------------------------------
# Quarantine stations
# ---------------------------------------

quarantine_stations = (
    station_routed
    .filter(col("DQ_CHG_002_check") == "FAIL")
    .drop("DQ_CHG_002_check")
)

# Define table names
TRUSTED_STATIONS = "workspace.silver_layer.trusted_stations"
QUAR_STATIONS = "workspace.silver_layer.quarantine_stations"

# Write outputs
trusted_stations.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(TRUSTED_STATIONS)

quarantine_stations.write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(QUAR_STATIONS)

print("Station routing completed.")

Station routing completed.


In [0]:
print("Candidate:", stations.count())
print("Trusted:", spark.table(TRUSTED_STATIONS).count())
print("Quarantine:", spark.table(QUAR_STATIONS).count())

Candidate: 180
Trusted: 178
Quarantine: 2


In [0]:
candidate_ids = stations.select("physical_record_id").distinct()

trusted_ids = (
    spark.table(TRUSTED_STATIONS)
    .select("physical_record_id")
    .distinct()
)

quarantine_ids = (
    spark.table(QUAR_STATIONS)
    .select("physical_record_id")
    .distinct()
)

candidate_count = candidate_ids.count()
trusted_count = trusted_ids.count()
quarantine_count = quarantine_ids.count()

overlap_count = (
    trusted_ids
    .join(quarantine_ids, "physical_record_id", "inner")
    .count()
)

variance = candidate_count - trusted_count - quarantine_count

print("Candidate distinct:", candidate_count)
print("Trusted distinct:", trusted_count)
print("Quarantine distinct:", quarantine_count)
print("Trusted/Quarantine overlap:", overlap_count)
print("Reconciliation variance:", variance)

Candidate distinct: 180
Trusted distinct: 178
Quarantine distinct: 2
Trusted/Quarantine overlap: 0
Reconciliation variance: 0


In [0]:
from pyspark.sql.functions import col, trim, to_date

chargers = spark.table(
    "workspace.silver_layer.silver_chargers_candidate"
)

trusted_stations_df = spark.table(
    TRUSTED_STATIONS
)

print("Charger candidate count:", chargers.count())
print("Trusted station count:", trusted_stations_df.count())

chargers.printSchema()

Charger candidate count: 1200
Trusted station count: 178
root
 |-- charger_id: string (nullable = true)
 |-- charger_label: string (nullable = true)
 |-- connector_position: long (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- firmware_major: long (nullable = true)
 |-- install_date: string (nullable = true)
 |-- manufacturer_band: string (nullable = true)
 |-- operational_status: string (nullable = true)
 |-- physical_record_id: string (nullable = true)
 |-- rated_power_kw: double (nullable = true)
 |-- smart_meter_enabled: boolean (nullable = true)
 |-- source_system: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)



In [0]:
charger_checks = (
    chargers.alias("c")
    .join(
        trusted_stations_df
        .select("station_id")
        .distinct()
        .alias("s"),
        col("c.station_id") == col("s.station_id"),
        "left"
    )
    .withColumn(
        "DQ_CHG_001_check",
        when(
            col("c.charger_id").isNull()
            | (trim(col("c.charger_id")) == "")
            | col("c.station_id").isNull()
            | (trim(col("c.station_id")) == "")
            | col("s.station_id").isNull()
            | col("c.connector_position").isNull()
            | (col("c.connector_position") <= 0)
            | col("c.connector_type").isNull()
            | (trim(col("c.connector_type")) == "")
            | col("c.rated_power_kw").isNull()
            | (col("c.rated_power_kw") <= 0)
            | col("c.install_date").isNull()
            | (trim(col("c.install_date")) == "")
            | col("c.operational_status").isNull()
            | (trim(col("c.operational_status")) == ""),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
charger_checks.groupBy(
    "DQ_CHG_001_check"
).count().show()

+----------------+-----+
|DQ_CHG_001_check|count|
+----------------+-----+
|            PASS| 1177|
|            FAIL|   23|
+----------------+-----+



In [0]:
charger_checks.filter(
    col("DQ_CHG_001_check") == "FAIL"
).select(
    "physical_record_id",
    "charger_id",
    "c.station_id",
    "connector_position",
    "connector_type",
    "rated_power_kw",
    "install_date",
    "operational_status"
).show(20, truncate=False)

+------------------+----------+----------+------------------+--------------+--------------+------------+------------------+
|physical_record_id|charger_id|station_id|connector_position|connector_type|rated_power_kw|install_date|operational_status|
+------------------+----------+----------+------------------+--------------+--------------+------------+------------------+
|CHGREC0001155     |CHG01155  |STN9999   |1                 |BHARAT_AC001  |3.3           |2023-10-07  |ACTIVE            |
|CHGREC0001159     |CHG01159  |STN9999   |5                 |BHARAT_DC001  |15.0          |2024-09-02  |ACTIVE            |
|CHGREC0001160     |CHG01160  |STN9999   |6                 |TYPE2_AC      |22.0          |2023-12-29  |ACTIVE            |
|CHGREC0001166     |CHG01166  |STN0176   |1                 |CCS2_DC       |999.0         |2023-01-10  |MAINTENANCE       |
|CHGREC0001167     |CHG01167  |STN9999   |2                 |CCS2_DC       |60.0          |2024-03-15  |OFFLINE           |
|CHGREC0

In [0]:
charger_checks.filter(
    col("DQ_CHG_001_check") == "FAIL"
).select(
    "physical_record_id",
    "charger_id",
    "c.station_id",
    "connector_position",
    "connector_type",
    "rated_power_kw",
    "install_date",
    "operational_status"
).show(30, truncate=False)

+------------------+----------+----------+------------------+--------------+--------------+------------+------------------+
|physical_record_id|charger_id|station_id|connector_position|connector_type|rated_power_kw|install_date|operational_status|
+------------------+----------+----------+------------------+--------------+--------------+------------+------------------+
|CHGREC0001155     |CHG01155  |STN9999   |1                 |BHARAT_AC001  |3.3           |2023-10-07  |ACTIVE            |
|CHGREC0001159     |CHG01159  |STN9999   |5                 |BHARAT_DC001  |15.0          |2024-09-02  |ACTIVE            |
|CHGREC0001160     |CHG01160  |STN9999   |6                 |TYPE2_AC      |22.0          |2023-12-29  |ACTIVE            |
|CHGREC0001166     |CHG01166  |STN0176   |1                 |CCS2_DC       |999.0         |2023-01-10  |MAINTENANCE       |
|CHGREC0001167     |CHG01167  |STN9999   |2                 |CCS2_DC       |60.0          |2024-03-15  |OFFLINE           |
|CHGREC0

In [0]:
charger_fail_reasons = (
    charger_checks
    .withColumn(
        "fail_station_reference",
        when(
            col("c.station_id").isNull()
            | (trim(col("c.station_id")) == "")
            | col("s.station_id").isNull(),
            "station_reference"
        )
    )
    .withColumn(
        "fail_charger_id",
        when(
            col("charger_id").isNull()
            | (trim(col("charger_id")) == ""),
            "charger_id"
        )
    )
    .withColumn(
        "fail_connector_position",
        when(
            col("connector_position").isNull()
            | (col("connector_position") <= 0),
            "connector_position"
        )
    )
    .withColumn(
        "fail_connector_type",
        when(
            col("connector_type").isNull()
            | (trim(col("connector_type")) == ""),
            "connector_type"
        )
    )
    .withColumn(
        "fail_power",
        when(
            col("rated_power_kw").isNull()
            | (col("rated_power_kw") <= 0),
            "rated_power_kw"
        )
    )
    .withColumn(
        "fail_install_date",
        when(
            col("install_date").isNull()
            | (trim(col("install_date")) == ""),
            "install_date"
        )
    )
    .withColumn(
        "fail_status",
        when(
            col("operational_status").isNull()
            | (trim(col("operational_status")) == ""),
            "operational_status"
        )
    )
)

charger_fail_reasons.filter(
    col("DQ_CHG_001_check") == "FAIL"
).select(
    "physical_record_id",
    "charger_id",
    "c.station_id",
    "rated_power_kw",
    "install_date",
    "operational_status",
    "fail_station_reference",
    "fail_charger_id",
    "fail_connector_position",
    "fail_connector_type",
    "fail_power",
    "fail_install_date",
    "fail_status"
).show(30, truncate=False)

+------------------+----------+----------+--------------+------------+------------------+----------------------+---------------+-----------------------+-------------------+----------+-----------------+-----------+
|physical_record_id|charger_id|station_id|rated_power_kw|install_date|operational_status|fail_station_reference|fail_charger_id|fail_connector_position|fail_connector_type|fail_power|fail_install_date|fail_status|
+------------------+----------+----------+--------------+------------+------------------+----------------------+---------------+-----------------------+-------------------+----------+-----------------+-----------+
|CHGREC0001155     |CHG01155  |STN9999   |3.3           |2023-10-07  |ACTIVE            |station_reference     |NULL           |NULL                   |NULL               |NULL      |NULL             |NULL       |
|CHGREC0001159     |CHG01159  |STN9999   |15.0          |2024-09-02  |ACTIVE            |station_reference     |NULL           |NULL            

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    trim,
    when,
    concat_ws,
    current_timestamp
)

charger_routed = (
    charger_checks
    .withColumn(
        "failure_count",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "failed_rule_ids",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("DQ-CHG-001")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "failure_reasons",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            concat_ws(
                "; ",
                when(
                    col("charger_id").isNull()
                    | (trim(col("charger_id")) == ""),
                    lit("charger_id is null or blank")
                ),
                when(
                    col("c.station_id").isNull()
                    | (trim(col("c.station_id")) == ""),
                    lit("station_id is null or blank")
                ),
                when(
                    col("connector_position").isNull()
                    | (col("connector_position") <= 0),
                    lit("connector_position is invalid")
                ),
                when(
                    col("connector_type").isNull()
                    | (trim(col("connector_type")) == ""),
                    lit("connector_type is null or blank")
                ),
                when(
                    col("rated_power_kw").isNull()
                    | (col("rated_power_kw") <= 0),
                    lit("rated_power_kw is invalid")
                ),
                when(
                    col("install_date").isNull()
                    | (trim(col("install_date")) == ""),
                    lit("install_date is null or blank")
                ),
                when(
                    col("operational_status").isNull()
                    | (trim(col("operational_status")) == ""),
                    lit("operational_status is null or blank")
                )
            )
        ).otherwise(
            lit(None).cast("string")
        )
    )
    .withColumn(
        "highest_severity",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("CRITICAL")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "dq_status",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("QUARANTINE")
        ).otherwise(lit("TRUSTED"))
    )
    .withColumn(
        "dq_run_id",
        lit(dq_run_id)
    )
    .withColumn(
        "dq_checked_ts",
        current_timestamp()
    )
    .withColumn(
        "rule_id",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("DQ-CHG-001")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "rule_name",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("Charger master integrity")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "severity",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("CRITICAL")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "rework_status",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            lit("PENDING")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "quarantined_at",
        when(
            col("DQ_CHG_001_check") == "FAIL",
            current_timestamp()
        ).otherwise(lit(None).cast("timestamp"))
    )
)

In [0]:
trusted_chargers = (
    charger_routed
    .filter(col("DQ_CHG_001_check") == "PASS")
    .drop("DQ_CHG_001_check")
)

quarantine_chargers = (
    charger_routed
    .filter(col("DQ_CHG_001_check") == "FAIL")
    .drop("DQ_CHG_001_check")
)

In [0]:
# Define table names
TRUSTED_CHARGERS = "workspace.silver_layer.trusted_chargers"
QUAR_CHARGERS = "workspace.silver_layer.quarantine_chargers"

# Drop duplicate station_id column from the join before writing
trusted_chargers = trusted_chargers.drop(col("s.station_id"))
quarantine_chargers = quarantine_chargers.drop(col("s.station_id"))

trusted_chargers.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TRUSTED_CHARGERS)

quarantine_chargers.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(QUAR_CHARGERS)

print("Charger DQ routing completed.")

Charger DQ routing completed.


In [0]:
candidate_ids = chargers.select(
    "physical_record_id"
).distinct()

trusted_ids = spark.table(
    TRUSTED_CHARGERS
).select(
    "physical_record_id"
).distinct()

quarantine_ids = spark.table(
    QUAR_CHARGERS
).select(
    "physical_record_id"
).distinct()

candidate_count = candidate_ids.count()
trusted_count = trusted_ids.count()
quarantine_count = quarantine_ids.count()

overlap_count = (
    trusted_ids
    .join(
        quarantine_ids,
        "physical_record_id",
        "inner"
    )
    .count()
)

variance = (
    candidate_count
    - trusted_count
    - quarantine_count
)

print("Candidate distinct:", candidate_count)
print("Trusted distinct:", trusted_count)
print("Quarantine distinct:", quarantine_count)
print("Trusted/Quarantine overlap:", overlap_count)
print("Reconciliation variance:", variance)

Candidate distinct: 1200
Trusted distinct: 1177
Quarantine distinct: 23
Trusted/Quarantine overlap: 0
Reconciliation variance: 0


In [0]:
sessions = spark.table("workspace.silver_layer.silver_sessions_candidate")
sessions.printSchema()
sessions.show(5, truncate=False)

root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

In [0]:
from pyspark.sql.functions import col, trim, when, lit

sessions = spark.table(
    "workspace.silver_layer.silver_sessions_candidate"
)

trusted_stations_df = spark.table(
    "workspace.silver_layer.trusted_stations"
)

trusted_chargers_df = spark.table(
    "workspace.silver_layer.trusted_chargers"
)

print("Session candidate count:", sessions.count())
print("Trusted stations:", trusted_stations_df.count())
print("Trusted chargers:", trusted_chargers_df.count())

sessions.printSchema()

Session candidate count: 300000
Trusted stations: 178
Trusted chargers: 1177
root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion

In [0]:
session_identity_checks = (
    sessions
    .withColumn(
        "DQ_SES_001_check",
        when(
            col("session_id").isNull()
            | (trim(col("session_id")) == ""),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
session_identity_checks.groupBy(
    "DQ_SES_001_check"
).count().show()

+----------------+------+
|DQ_SES_001_check| count|
+----------------+------+
|            PASS|299600|
|            FAIL|   400|
+----------------+------+



In [0]:
session_identity_checks.filter(
    col("DQ_SES_001_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "station_id",
    "charger_id",
    "arrival_ts",
    "charge_start_ts",
    "charge_end_ts",
    "departure_ts"
).show(20, truncate=False)

+------------------+----------+----------+----------+-----------------------+-----------------------+-----------------------+-----------------------+
|physical_record_id|session_id|station_id|charger_id|arrival_ts             |charge_start_ts        |charge_end_ts          |departure_ts           |
+------------------+----------+----------+----------+-----------------------+-----------------------+-----------------------+-----------------------+
|SESREC000000223   |NULL      |STN0001   |CHG00001  |2026-03-28 07:09:22.244|2026-03-28 07:18:22.244|2026-03-28 08:34:22.244|2026-03-28 08:49:22.244|
|SESREC000002346   |NULL      |STN0002   |CHG00010  |2026-03-08 15:00:51.721|2026-03-08 15:11:51.721|2026-03-08 15:57:51.721|2026-03-08 16:00:51.721|
|SESREC000002854   |NULL      |STN0002   |CHG00012  |2026-02-27 08:04:00.415|2026-02-27 08:16:00.415|2026-02-27 10:21:00.415|2026-02-27 10:28:00.415|
|SESREC000002903   |NULL      |STN0002   |CHG00012  |2026-03-16 23:58:53.78 |2026-03-16 23:58:53.78 

In [0]:
session_ref_checks = (
    session_identity_checks
    .alias("ses")
    .join(
        trusted_stations_df
        .select("station_id")
        .distinct()
        .alias("stn"),
        col("ses.station_id") == col("stn.station_id"),
        "left"
    )
    .join(
        trusted_chargers_df
        .select(
            "charger_id",
            "station_id"
        )
        .distinct()
        .alias("chg"),
        col("ses.charger_id") == col("chg.charger_id"),
        "left"
    )
    .withColumn(
        "DQ_SES_002_check",
        when(
            col("ses.station_id").isNull()
            | (trim(col("ses.station_id")) == "")
            | col("ses.charger_id").isNull()
            | (trim(col("ses.charger_id")) == "")
            | col("stn.station_id").isNull()
            | col("chg.charger_id").isNull()
            | (col("chg.station_id") != col("ses.station_id")),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
session_ref_checks.groupBy(
    "DQ_SES_002_check"
).count().show()

+----------------+------+
|DQ_SES_002_check| count|
+----------------+------+
|            PASS|299100|
|            FAIL|   900|
+----------------+------+



In [0]:
session_ref_checks.filter(
    col("DQ_SES_002_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "ses.station_id",
    "ses.charger_id",
    "chg.station_id"
).show(20, truncate=False)

+------------------+------------+----------+----------+----------+
|physical_record_id|session_id  |station_id|charger_id|station_id|
+------------------+------------+----------+----------+----------+
|SESREC000000149   |SES000000149|STN0002   |CHG00001  |STN0001   |
|SESREC000000338   |SES000000338|STN0001   |CHG99999  |NULL      |
|SESREC000000398   |SES000000398|STN0002   |CHG00002  |STN0001   |
|SESREC000000655   |SES000000655|STN0001   |CHG99999  |NULL      |
|SESREC000000689   |SES000000689|STN9999   |CHG00003  |STN0001   |
|SESREC000001195   |SES000001195|STN0002   |CHG00005  |STN0001   |
|SESREC000001271   |SES000001271|STN0002   |CHG99999  |NULL      |
|SESREC000001472   |SES000001472|STN0002   |CHG99999  |NULL      |
|SESREC000002079   |SES000002079|STN9999   |CHG00009  |STN0002   |
|SESREC000002263   |SES000002263|STN0002   |CHG99999  |NULL      |
|SESREC000003142   |SES000003142|STN9999   |CHG00013  |STN0002   |
|SESREC000003280   |SES000003280|STN0002   |CHG99999  |NULL   

In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    lit,
    when
)

session_chronology_checks = (
    session_ref_checks
    .withColumn(
        "DQ_SES_003_check",
        when(
            col("arrival_ts").isNull()
            | col("charge_start_ts").isNull()
            | col("charge_end_ts").isNull()
            | col("departure_ts").isNull()
            | (col("charge_start_ts") < col("arrival_ts"))
            | (col("charge_end_ts") < col("charge_start_ts"))
            | (col("departure_ts") < col("charge_end_ts"))
            | (to_date(col("arrival_ts")) < lit("2026-01-01"))
            | (to_date(col("arrival_ts")) > lit("2026-03-31")),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
session_chronology_checks.groupBy(
    "DQ_SES_003_check"
).count().show()

+----------------+------+
|DQ_SES_003_check| count|
+----------------+------+
|            PASS|289996|
|            FAIL| 10004|
+----------------+------+



In [0]:
session_chronology_checks.filter(
    col("DQ_SES_003_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "ses.station_id",
    "ses.charger_id",
    "arrival_ts",
    "charge_start_ts",
    "charge_end_ts",
    "departure_ts",
    "batch_date"
).show(20, truncate=False)

+------------------+------------+----------+----------+-----------------------+-----------------------+-----------------------+-----------------------+----------+
|physical_record_id|session_id  |station_id|charger_id|arrival_ts             |charge_start_ts        |charge_end_ts          |departure_ts           |batch_date|
+------------------+------------+----------+----------+-----------------------+-----------------------+-----------------------+-----------------------+----------+
|SESREC000000014   |SES000000014|STN0001   |CHG00001  |2026-01-06 02:26:19.75 |NULL                   |NULL                   |2026-01-06 02:50:19.75 |2026-01-06|
|SESREC000000020   |SES000000020|STN0001   |CHG00001  |2026-01-08 10:26:32.425|NULL                   |NULL                   |2026-01-08 10:49:32.425|2026-01-08|
|SESREC000000068   |SES000000068|STN0001   |CHG00001  |2026-01-27 02:00:14.972|2026-01-27 02:00:14.972|2026-01-27 04:10:14.972|2026-01-27 01:55:14.972|2026-01-27|
|SESREC000000071   |SE

In [0]:
session_lifecycle_checks = (
    session_chronology_checks
    .withColumn(
        "DQ_SES_004_check",
        when(
            # COMPLETED sessions
            (
                (col("final_status") == "COMPLETED")
                & (
                    col("charge_start_ts").isNull()
                    | col("charge_end_ts").isNull()
                    | col("departure_ts").isNull()
                    | col("end_reason").isNull()
                    | (trim(col("end_reason")) == "")
                    | col("energy_kwh").isNull()
                )
            )
            |
            # INTERRUPTED / CANCELLED sessions
            (
                col("final_status").isin(
                    "INTERRUPTED",
                    "CANCELLED"
                )
                & (
                    col("departure_ts").isNull()
                    | col("end_reason").isNull()
                    | (trim(col("end_reason")) == "")
                )
            ),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
session_lifecycle_checks.filter(
    col("DQ_SES_004_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "final_status",
    "end_reason",
    "arrival_ts",
    "charge_start_ts",
    "charge_end_ts",
    "departure_ts",
    "energy_kwh"
).show(20, truncate=False)

+------------------+------------+------------+---------------+-----------------------+-----------------------+-------------+-----------------------+----------+
|physical_record_id|session_id  |final_status|end_reason     |arrival_ts             |charge_start_ts        |charge_end_ts|departure_ts           |energy_kwh|
+------------------+------------+------------+---------------+-----------------------+-----------------------+-------------+-----------------------+----------+
|SESREC000000227   |SES000000227|COMPLETED   |TARGET_REACHED |2026-03-29 20:48:39.149|2026-03-29 20:50:39.149|NULL         |2026-03-29 21:47:39.149|11.615    |
|SESREC000000375   |SES000000375|COMPLETED   |POWER_LIMIT    |2026-02-15 17:10:21.97 |2026-02-15 17:12:21.97 |NULL         |2026-02-15 17:34:21.97 |23.225    |
|SESREC000000462   |SES000000462|COMPLETED   |VEHICLE_REQUEST|2026-03-15 11:45:06.859|2026-03-15 11:48:06.859|NULL         |2026-03-15 12:03:06.859|20.327    |
|SESREC000000843   |SES000000843|COMPLET

In [0]:
session_lifecycle_checks.groupBy(
    "DQ_SES_004_check"
).count().show()

+----------------+------+
|DQ_SES_004_check| count|
+----------------+------+
|            PASS|299798|
|            FAIL|   202|
+----------------+------+



In [0]:
from pyspark.sql.functions import col, expr

# Select distinct columns to avoid ambiguity in self-join
base_for_overlap = session_lifecycle_checks.select(
    "physical_record_id",
    "session_id",
    col("ses.charger_id").alias("charger_id"),
    "charge_start_ts",
    "charge_end_ts"
)

s1 = base_for_overlap.alias("s1")
s2 = base_for_overlap.alias("s2")

actual_overlaps = (
    s1.join(
        s2,
        (col("s1.charger_id") == col("s2.charger_id"))
        & (col("s1.physical_record_id") != col("s2.physical_record_id"))
        & col("s1.charge_start_ts").isNotNull()
        & col("s1.charge_end_ts").isNotNull()
        & col("s2.charge_start_ts").isNotNull()
        & col("s2.charge_end_ts").isNotNull()
        & (col("s1.charge_start_ts") < col("s2.charge_end_ts"))
        & (col("s2.charge_start_ts") < col("s1.charge_end_ts")),
        "left"
    )
    .select(
        col("s1.physical_record_id").alias("physical_record_id"),
        col("s1.session_id").alias("session_id"),
        col("s1.charger_id").alias("charger_id")
    )
    .distinct()
    .withColumn("DQ_SES_005_overlap", lit("FAIL"))
)

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
session_overlap_checks = (
    session_lifecycle_checks
    .join(
        actual_overlaps.select(
            "physical_record_id",
            "DQ_SES_005_overlap"
        ),
        on="physical_record_id",
        how="left"
    )
    .withColumn(
        "DQ_SES_005_overlap",
        when(
            col("DQ_SES_005_overlap").isNull(),
            "PASS"
        ).otherwise(col("DQ_SES_005_overlap"))
    )
)

In [0]:
session_overlap_checks.groupBy(
    "DQ_SES_005_overlap"
).count().show()

+------------------+------+
|DQ_SES_005_overlap| count|
+------------------+------+
|              FAIL|300000|
+------------------+------+



In [0]:
session_overlap_checks.filter(
    col("DQ_SES_005_overlap") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "ses.station_id",
    "ses.charger_id",
    "charge_start_ts",
    "charge_end_ts"
).show(20, truncate=False)

+------------------+------------+----------+----------+-----------------------+-----------------------+
|physical_record_id|session_id  |station_id|charger_id|charge_start_ts        |charge_end_ts          |
+------------------+------------+----------+----------+-----------------------+-----------------------+
|SESREC000000001   |SES000000001|STN0001   |CHG00001  |2026-01-01 00:50:49.093|2026-01-01 01:20:49.093|
|SESREC000000002   |SES000000002|STN0001   |CHG00001  |2026-01-01 10:26:00.23 |2026-01-01 11:20:00.23 |
|SESREC000000003   |SES000000003|STN0001   |CHG00001  |2026-01-01 19:03:23.379|2026-01-01 21:08:23.379|
|SESREC000000004   |SES000000004|STN0001   |CHG00001  |2026-01-02 05:27:46.072|2026-01-02 06:51:46.072|
|SESREC000000005   |SES000000005|STN0001   |CHG00001  |2026-01-02 13:53:24.357|2026-01-02 15:21:24.357|
|SESREC000000006   |SES000000006|STN0001   |CHG00001  |2026-01-03 00:02:01.749|2026-01-03 02:06:01.749|
|SESREC000000007   |SES000000007|STN0001   |CHG00001  |2026-01-0

In [0]:
from pyspark.sql.functions import col, lit

a = (
    session_lifecycle_checks
    .filter(
        (col("DQ_SES_003_check") == "PASS")
        & col("charge_start_ts").isNotNull()
        & col("charge_end_ts").isNotNull()
    )
    .select(
        "physical_record_id",
        "session_id",
        col("ses.station_id").alias("station_id"),
        col("ses.charger_id").alias("charger_id"),
        "charge_start_ts",
        "charge_end_ts"
    )
    .alias("a")
)

b = (
    session_lifecycle_checks
    .filter(
        (col("DQ_SES_003_check") == "PASS")
        & col("charge_start_ts").isNotNull()
        & col("charge_end_ts").isNotNull()
    )
    .select(
        "physical_record_id",
        col("ses.charger_id").alias("charger_id"),
        "charge_start_ts",
        "charge_end_ts"
    )
    .alias("b")
)

overlap_ids = (
    a.join(
        b,
        (col("a.charger_id") == col("b.charger_id"))
        & (col("a.physical_record_id") != col("b.physical_record_id"))
        & (col("b.charge_start_ts") < col("a.charge_start_ts"))
        & (col("b.charge_end_ts") > col("a.charge_start_ts")),
        "inner"
    )
    .select(
        col("a.physical_record_id").alias("physical_record_id")
    )
    .distinct()
    .withColumn("DQ_SES_005_overlap", lit("FAIL"))
)

In [0]:
session_overlap_checks = (
    session_lifecycle_checks
    .join(
        overlap_ids,
        on="physical_record_id",
        how="left"
    )
    .withColumn(
        "DQ_SES_005_overlap",
        when(
            col("DQ_SES_005_overlap").isNull(),
            "PASS"
        ).otherwise(col("DQ_SES_005_overlap"))
    )
)

In [0]:
from pyspark.sql.functions import when

In [0]:
session_overlap_checks.groupBy(
    "DQ_SES_005_overlap"
).count().show()

+------------------+------+
|DQ_SES_005_overlap| count|
+------------------+------+
|              PASS|298768|
|              FAIL|  1232|
+------------------+------+



In [0]:
session_overlap_checks.filter(
    col("DQ_SES_005_overlap") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "ses.station_id",
    "ses.charger_id",
    "charge_start_ts",
    "charge_end_ts"
).show(20, truncate=False)

+------------------+------------+----------+----------+-----------------------+-----------------------+
|physical_record_id|session_id  |station_id|charger_id|charge_start_ts        |charge_end_ts          |
+------------------+------------+----------+----------+-----------------------+-----------------------+
|SESREC000000338   |SES000000338|STN0001   |CHG99999  |2026-02-03 21:32:00.534|2026-02-03 22:02:00.534|
|SESREC000000982   |SES000000982|STN0001   |CHG00004  |2026-03-18 14:03:03.975|2026-03-18 15:14:03.975|
|SESREC000001232   |SES000001232|STN0001   |CHG00005  |2026-03-24 12:19:50.47 |2026-03-24 14:14:50.47 |
|SESREC000001233   |SES000001233|STN0001   |CHG00005  |2026-03-24 22:16:28.131|2026-03-24 23:37:28.131|
|SESREC000001364   |SES000001364|STN0002   |CHG00006  |2026-02-14 21:45:05.013|2026-02-14 22:39:05.013|
|SESREC000001512   |SES000001512|STN0002   |CHG00007  |2026-01-16 00:38:11.734|2026-01-16 04:12:11.734|
|SESREC000002439   |SES000002439|STN0002   |CHG00011  |2026-01-1

In [0]:
station_capacity_df = (
    trusted_stations_df
    .select(
        col("station_id").alias("capacity_station_id"),
        "connector_capacity"
    )
    .distinct()
)

session_capacity_base = (
    session_overlap_checks
    .join(
        station_capacity_df,
        col("ses.station_id") == col("capacity_station_id"),
        how="left"
    )
)

In [0]:
a = (
    session_capacity_base
    .filter(
        (col("DQ_SES_003_check") == "PASS")
        & col("charge_start_ts").isNotNull()
        & col("charge_end_ts").isNotNull()
        & col("connector_capacity").isNotNull()
    )
    .select(
        "physical_record_id",
        col("ses.station_id").alias("station_id"),
        "charge_start_ts",
        "charge_end_ts",
        "connector_capacity"
    )
    .alias("a")
)

b = (
    session_capacity_base
    .filter(
        (col("DQ_SES_003_check") == "PASS")
        & col("charge_start_ts").isNotNull()
        & col("charge_end_ts").isNotNull()
    )
    .select(
        "physical_record_id",
        col("ses.station_id").alias("station_id"),
        "charge_start_ts",
        "charge_end_ts"
    )
    .alias("b")
)

station_concurrency = (
    a.join(
        b,
        (col("a.station_id") == col("b.station_id"))
        & (col("b.charge_start_ts") < col("a.charge_end_ts"))
        & (col("b.charge_end_ts") > col("a.charge_start_ts")),
        "inner"
    )
    .groupBy(
        col("a.physical_record_id").alias("physical_record_id"),
        col("a.station_id").alias("station_id"),
        col("a.connector_capacity").alias("connector_capacity")
    )
    .count()
    .withColumnRenamed("count", "concurrent_session_count")
)

In [0]:
capacity_failures = (
    station_concurrency
    .filter(
        col("concurrent_session_count") > col("connector_capacity")
    )
    .select(
        "physical_record_id",
        "station_id",
        "connector_capacity",
        "concurrent_session_count"
    )
    .distinct()
)

In [0]:
capacity_failures.count()

372

In [0]:
capacity_failures.show(20, truncate=False)

+------------------+----------+------------------+------------------------+
|physical_record_id|station_id|connector_capacity|concurrent_session_count|
+------------------+----------+------------------+------------------------+
|SESREC000032825   |STN0019   |5                 |16                      |
|SESREC000037013   |STN0023   |5                 |15                      |
|SESREC000029794   |STN0017   |9                 |30                      |
|SESREC000029228   |STN0017   |9                 |10                      |
|SESREC000028174   |STN0017   |9                 |10                      |
|SESREC000028455   |STN0017   |9                 |10                      |
|SESREC000029212   |STN0017   |9                 |10                      |
|SESREC000030050   |STN0017   |9                 |30                      |
|SESREC000029957   |STN0017   |9                 |29                      |
|SESREC000031930   |STN0019   |5                 |15                      |
|SESREC00002

In [0]:
from pyspark.sql.functions import col, lit, when, concat_ws

capacity_failure_ids = (
    capacity_failures
    .select("physical_record_id")
    .distinct()
    .withColumn("DQ_SES_005_capacity", lit("FAIL"))
)

session_dq005_checks = (
    session_overlap_checks
    .join(
        capacity_failure_ids,
        on="physical_record_id",
        how="left"
    )
    .withColumn(
        "DQ_SES_005_check",
        when(
            (col("DQ_SES_005_overlap") == "FAIL") |
            (col("DQ_SES_005_capacity") == "FAIL"),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn(
        "DQ_SES_005_reason",
        concat_ws(
            "; ",
            when(
                col("DQ_SES_005_overlap") == "FAIL",
                lit("Overlapping sessions on same charger")
            ),
            when(
                col("DQ_SES_005_capacity") == "FAIL",
                lit("Concurrent sessions exceed station connector capacity")
            )
        )
    )
)

In [0]:
session_dq005_checks.groupBy("DQ_SES_005_check").count().show()

+----------------+------+
|DQ_SES_005_check| count|
+----------------+------+
|            PASS|298407|
|            FAIL|  1593|
+----------------+------+



In [0]:
session_dq005_checks \
    .filter(col("DQ_SES_005_check") == "FAIL") \
    .select(
        "physical_record_id",
        "session_id",
        "ses.station_id",
        "ses.charger_id",
        "DQ_SES_005_overlap",
        "DQ_SES_005_capacity",
        "DQ_SES_005_reason"
    ) \
    .show(20, truncate=False)

+------------------+------------+----------+----------+------------------+-------------------+-----------------------------------------------------+
|physical_record_id|session_id  |station_id|charger_id|DQ_SES_005_overlap|DQ_SES_005_capacity|DQ_SES_005_reason                                    |
+------------------+------------+----------+----------+------------------+-------------------+-----------------------------------------------------+
|SESREC000009122   |SES000009122|STN0005   |CHG00036  |PASS              |FAIL               |Concurrent sessions exceed station connector capacity|
|SESREC000011222   |SES000011222|STN0007   |CHG00044  |FAIL              |NULL               |Overlapping sessions on same charger                 |
|SESREC000050268   |SES000050268|STN0030   |CHG00192  |FAIL              |NULL               |Overlapping sessions on same charger                 |
|SESREC000064112   |SES000064112|STN0038   |CHG00245  |PASS              |FAIL               |Concurrent s

In [0]:
session_dq005_checks.groupBy("DQ_SES_005_check").count().show()

+----------------+------+
|DQ_SES_005_check| count|
+----------------+------+
|            PASS|298407|
|            FAIL|  1593|
+----------------+------+



In [0]:
sessions.printSchema()

root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

In [0]:
sessions.select(
    "energy_kwh",
    "arrival_ts",
    "charge_start_ts",
    "charge_end_ts",
    "departure_ts"
).show(10, truncate=False)

+----------+-----------------------+-----------------------+-----------------------+-----------------------+
|energy_kwh|arrival_ts             |charge_start_ts        |charge_end_ts          |departure_ts           |
+----------+-----------------------+-----------------------+-----------------------+-----------------------+
|9.034     |2026-01-01 00:34:49.093|2026-01-01 00:50:49.093|2026-01-01 01:20:49.093|2026-01-01 01:30:49.093|
|13.419    |2026-01-01 10:26:00.23 |2026-01-01 10:26:00.23 |2026-01-01 11:20:00.23 |2026-01-01 11:39:00.23 |
|4.6       |2026-01-01 19:03:23.379|2026-01-01 19:03:23.379|2026-01-01 21:08:23.379|2026-01-01 21:27:23.379|
|28.076    |2026-01-02 05:23:46.072|2026-01-02 05:27:46.072|2026-01-02 06:51:46.072|2026-01-02 06:59:46.072|
|29.718    |2026-01-02 13:51:24.357|2026-01-02 13:53:24.357|2026-01-02 15:21:24.357|2026-01-02 15:31:24.357|
|11.04     |2026-01-02 23:52:01.749|2026-01-03 00:02:01.749|2026-01-03 02:06:01.749|2026-01-03 02:15:01.749|
|10.847    |2026-01

In [0]:
from pyspark.sql.functions import col, when, unix_timestamp

session_measures = (
    session_dq005_checks
    .withColumn(
        "duration_minutes",
        (
            unix_timestamp("charge_end_ts")
            - unix_timestamp("charge_start_ts")
        ) / 60.0
    )
    .withColumn(
        "occupied_minutes",
        (
            unix_timestamp("departure_ts")
            - unix_timestamp("arrival_ts")
        ) / 60.0
    )
)

In [0]:
session_measures = (
    session_measures
    .withColumn(
        "DQ_SES_006_check",
        when(
            col("energy_kwh").isNull()
            | (col("energy_kwh") < 0)
            | col("duration_minutes").isNull()
            | (col("duration_minutes") < 0)
            | col("occupied_minutes").isNull()
            | (col("occupied_minutes") < 0),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
session_measures.groupBy("DQ_SES_006_check").count().show()

+----------------+------+
|DQ_SES_006_check| count|
+----------------+------+
|            PASS|289945|
|            FAIL| 10055|
+----------------+------+



In [0]:
session_measures \
    .filter(col("DQ_SES_006_check") == "FAIL") \
    .select(
        "physical_record_id",
        "session_id",
        "energy_kwh",
        "arrival_ts",
        "charge_start_ts",
        "charge_end_ts",
        "departure_ts",
        "duration_minutes",
        "occupied_minutes"
    ) \
    .show(20, truncate=False)

+------------------+------------+----------+-----------------------+-----------------------+-----------------------+-----------------------+----------------+----------------+
|physical_record_id|session_id  |energy_kwh|arrival_ts             |charge_start_ts        |charge_end_ts          |departure_ts           |duration_minutes|occupied_minutes|
+------------------+------------+----------+-----------------------+-----------------------+-----------------------+-----------------------+----------------+----------------+
|SESREC000000014   |SES000000014|0.0       |2026-01-06 02:26:19.75 |NULL                   |NULL                   |2026-01-06 02:50:19.75 |NULL            |24.0            |
|SESREC000000020   |SES000000020|0.0       |2026-01-08 10:26:32.425|NULL                   |NULL                   |2026-01-08 10:49:32.425|NULL            |23.0            |
|SESREC000000068   |SES000000068|36.914    |2026-01-27 02:00:14.972|2026-01-27 02:00:14.972|2026-01-27 04:10:14.972|2026-01-2

In [0]:
session_measures.printSchema()

root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

In [0]:
trusted_chargers_df.select(
    "charger_id",
    "rated_power_kw"
).show(10, truncate=False)

+----------+--------------+
|charger_id|rated_power_kw|
+----------+--------------+
|CHG00001  |22.0          |
|CHG00002  |150.0         |
|CHG00003  |60.0          |
|CHG00004  |3.3           |
|CHG00005  |15.0          |
|CHG00006  |7.4           |
|CHG00007  |7.4           |
|CHG00008  |11.0          |
|CHG00009  |50.0          |
|CHG00010  |22.0          |
+----------+--------------+
only showing top 10 rows


In [0]:
spark.sql("SHOW TABLES IN workspace.silver_layer").show(100, truncate=False)

+------------+----------------------------+-----------+
|database    |tableName                   |isTemporary|
+------------+----------------------------+-----------+
|silver_layer|quarantine_chargers         |false      |
|silver_layer|quarantine_stations         |false      |
|silver_layer|silver_chargers_candidate   |false      |
|silver_layer|silver_maintenance_candidate|false      |
|silver_layer|silver_sessions_candidate   |false      |
|silver_layer|silver_stations_candidate   |false      |
|silver_layer|trusted_chargers            |false      |
|silver_layer|trusted_stations            |false      |
+------------+----------------------------+-----------+



In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(100, truncate=False)

+--------+--------------------------------------+-----------+
|database|tableName                             |isTemporary|
+--------+--------------------------------------+-----------+
|default |bronze_chargers                       |false      |
|default |bronze_maintenance                    |false      |
|default |bronze_sessions                       |false      |
|default |bronze_stations                       |false      |
|default |quarantine_chargeiq_chargers          |false      |
|default |quarantine_chargeiq_maintenance       |false      |
|default |quarantine_chargeiq_sessions          |false      |
|default |quarantine_chargeiq_stations          |false      |
|default |shiptrack_week03_bronze_demo_shipments|false      |
|default |shiptrack_week03_lineage_demo_view    |false      |
|default |trusted_silver_chargeiq_chargers      |false      |
|default |trusted_silver_chargeiq_maintenance   |false      |
|default |trusted_silver_chargeiq_sessions      |false      |
|default

In [0]:
trusted_chargers_df.select(
    "rated_power_kw"
).summary().show()

+-------+------------------+
|summary|    rated_power_kw|
+-------+------------------+
|  count|              1177|
|   mean|50.662192013594044|
| stddev| 70.34809639414146|
|    min|               3.3|
|    25%|              15.0|
|    50%|              30.0|
|    75%|              60.0|
|    max|             999.0|
+-------+------------------+



In [0]:
trusted_chargers_df.groupBy(
    "rated_power_kw"
).count().orderBy(
    "rated_power_kw"
).show(50, truncate=False)

+--------------+-----+
|rated_power_kw|count|
+--------------+-----+
|3.3           |52   |
|7.4           |142  |
|11.0          |79   |
|15.0          |60   |
|22.0          |106  |
|30.0          |225  |
|50.0          |168  |
|60.0          |146  |
|120.0         |95   |
|150.0         |100  |
|999.0         |4    |
+--------------+-----+



In [0]:
from pyspark.sql.functions import col, when

session_power = (
    session_measures
    .join(
        trusted_chargers_df.select(
            "charger_id",
            "rated_power_kw"
        ).distinct(),
        on="charger_id",
        how="left"
    )
    .withColumn(
        "expected_energy_kwh",
        (
            col("rated_power_kw")
            * col("duration_minutes")
            / 60.0
        )
    )
    .withColumn(
        "energy_ratio",
        when(
            col("expected_energy_kwh") > 0,
            col("energy_kwh") / col("expected_energy_kwh")
        )
    )
)

In [0]:
session_power.select(
    "energy_kwh",
    "rated_power_kw",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).filter(
    col("energy_ratio").isNotNull()
).summary().show()

+-------+------------------+-----------------+-----------------+-------------------+-------------------+
|summary|        energy_kwh|   rated_power_kw| duration_minutes|expected_energy_kwh|       energy_ratio|
+-------+------------------+-----------------+-----------------+-------------------+-------------------+
|  count|            290187|           290187|           290187|             290187|             290187|
|   mean| 25.44854103043689|50.31253088529644|73.43724908421122| 44.047324104801504| 0.6580298544851759|
| stddev|19.539243948911928|43.89713683382583|64.51258480803028| 60.289864083924705|0.31580534475080657|
|    min|              -3.5|              3.3|              3.0|0.16499999999999998| -7.070707070707071|
|    25%|             9.323|             22.0|             39.0|               20.0|              0.576|
|    50%|              20.2|             30.0|             61.0| 35.833333333333336|  0.740025974025974|
|    75%|            40.488|             60.0|         

In [0]:
session_power.filter(
    col("energy_ratio").isNotNull()
).select(
    "physical_record_id",
    "session_id",
    "ses.charger_id",
    "rated_power_kw",
    "energy_kwh",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).orderBy(
    col("energy_ratio").desc()
).show(20, truncate=False)

+------------------+------------+----------+--------------+----------+----------------+-------------------+------------------+
|physical_record_id|session_id  |charger_id|rated_power_kw|energy_kwh|duration_minutes|expected_energy_kwh|energy_ratio      |
+------------------+------------+----------+--------------+----------+----------------+-------------------+------------------+
|SESREC000008181   |SES000008181|CHG00032  |3.3           |7.229     |3.0             |0.16499999999999998|43.81212121212122 |
|SESREC000186668   |SES000186668|CHG00716  |60.0          |48.872    |3.0             |3.0                |16.290666666666667|
|SESREC000205776   |SES000205776|CHG00788  |3.3           |50.0      |65.0            |3.575              |13.986013986013985|
|SESREC000214918   |SES000214918|CHG00822  |7.4           |50.0      |31.0            |3.8233333333333333 |13.077593722755013|
|SESREC000270177   |SES000270177|CHG01037  |3.3           |50.0      |71.0            |3.905              |12.8

In [0]:
session_power.filter(
    col("energy_ratio").isNotNull()
).select(
    "physical_record_id",
    "session_id",
    "ses.charger_id",
    "rated_power_kw",
    "energy_kwh",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).orderBy(
    col("energy_ratio").asc()
).show(20, truncate=False)

+------------------+------------+----------+--------------+----------+----------------+-------------------+--------------------+
|physical_record_id|session_id  |charger_id|rated_power_kw|energy_kwh|duration_minutes|expected_energy_kwh|energy_ratio        |
+------------------+------------+----------+--------------+----------+----------------+-------------------+--------------------+
|SESREC000011870   |SES000011870|CHG00046  |3.3           |-3.5      |9.0             |0.495              |-7.070707070707071  |
|SESREC000126771   |SES000126771|CHG00489  |7.4           |-3.5      |13.0            |1.6033333333333333 |-2.182952182952183  |
|SESREC000299428   |SES000299428|CHG01149  |7.4           |-3.5      |23.0            |2.836666666666667  |-1.2338425381903642 |
|SESREC000067254   |SES000067254|CHG00258  |3.3           |-3.5      |82.0            |4.51               |-0.7760532150776054 |
|SESREC000147018   |SES000147018|CHG00566  |22.0          |-3.5      |13.0            |4.76666666

In [0]:
# Search all available tables/views for configuration-related objects
spark.sql("SHOW TABLES IN workspace.silver_layer").show(100, truncate=False)
spark.sql("SHOW TABLES IN workspace.default").show(100, truncate=False)

+------------+----------------------------+-----------+
|database    |tableName                   |isTemporary|
+------------+----------------------------+-----------+
|silver_layer|quarantine_chargers         |false      |
|silver_layer|quarantine_stations         |false      |
|silver_layer|silver_chargers_candidate   |false      |
|silver_layer|silver_maintenance_candidate|false      |
|silver_layer|silver_sessions_candidate   |false      |
|silver_layer|silver_stations_candidate   |false      |
|silver_layer|trusted_chargers            |false      |
|silver_layer|trusted_stations            |false      |
+------------+----------------------------+-----------+

+--------+--------------------------------------+-----------+
|database|tableName                             |isTemporary|
+--------+--------------------------------------+-----------+
|default |bronze_chargers                       |false      |
|default |bronze_maintenance                    |false      |
|default |bronze_

In [0]:
trusted_chargers_df.filter(
    col("rated_power_kw") == 999.0
).select(
    "charger_id",
    "station_id",
    "rated_power_kw",
    "operational_status",
    "connector_type"
).show(20, truncate=False)

+----------+----------+--------------+------------------+--------------+
|charger_id|station_id|rated_power_kw|operational_status|connector_type|
+----------+----------+--------------+------------------+--------------+
|CHG01163  |STN0175   |999.0         |ACTIVE            |TYPE2_AC      |
|CHG01186  |STN0179   |999.0         |OFFLINE           |TYPE2_AC      |
|CHG01191  |STN0179   |999.0         |MAINTENANCE       |TYPE2_AC      |
|CHG01194  |STN0179   |999.0         |OFFLINE           |TYPE2_AC      |
+----------+----------+--------------+------------------+--------------+



In [0]:
from pyspark.sql.functions import col, trim, count

# Find duplicate non-null/non-blank session IDs
duplicate_session_ids = (
    sessions
    .filter(
        col("session_id").isNotNull()
        & (trim(col("session_id")) != "")
    )
    .groupBy("session_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_session_ids.show(20, truncate=False)

print("Duplicate session_id values:", duplicate_session_ids.count())

+------------+-----+
|session_id  |count|
+------------+-----+
|SES000000008|2    |
|SES000017888|2    |
|SES000001643|2    |
|SES000001770|2    |
|SES000002180|2    |
|SES000003112|2    |
|SES000003908|2    |
|SES000004443|2    |
|SES000235890|2    |
|SES000086438|2    |
|SES000005316|2    |
|SES000006586|2    |
|SES000232604|2    |
|SES000007443|2    |
|SES000008558|2    |
|SES000008837|2    |
|SES000009940|2    |
|SES000054230|2    |
|SES000049247|2    |
|SES000010155|2    |
+------------+-----+
only showing top 20 rows
Duplicate session_id values: 400


In [0]:
from pyspark.sql.functions import col, trim, when, lit

duplicate_ids = duplicate_session_ids.select("session_id").distinct()

session_identity_checks = (
    sessions
    .join(
        duplicate_ids.withColumn("duplicate_session_id", lit(True)),
        on="session_id",
        how="left"
    )
    .withColumn(
        "DQ_SES_001_check",
        when(
            col("session_id").isNull()
            | (trim(col("session_id")) == "")
            | col("duplicate_session_id").isNotNull(),
            "FAIL"
        ).otherwise("PASS")
    )
    .drop("duplicate_session_id")
)

In [0]:
from pyspark.sql.functions import lit

In [0]:
from pyspark.sql.functions import col, trim, count

duplicate_session_ids = (
    sessions
    .filter(
        col("session_id").isNotNull()
        & (trim(col("session_id")) != "")
    )
    .groupBy("session_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_session_ids.show(20, truncate=False)

print("Duplicate session_id values:", duplicate_session_ids.count())

+------------+-----+
|session_id  |count|
+------------+-----+
|SES000000008|2    |
|SES000017888|2    |
|SES000001643|2    |
|SES000001770|2    |
|SES000002180|2    |
|SES000003112|2    |
|SES000003908|2    |
|SES000004443|2    |
|SES000235890|2    |
|SES000086438|2    |
|SES000005316|2    |
|SES000006586|2    |
|SES000232604|2    |
|SES000007443|2    |
|SES000008558|2    |
|SES000008837|2    |
|SES000009940|2    |
|SES000054230|2    |
|SES000049247|2    |
|SES000010155|2    |
+------------+-----+
only showing top 20 rows
Duplicate session_id values: 400


In [0]:
from pyspark.sql.functions import col, trim, when, lit

duplicate_ids = (
    duplicate_session_ids
    .select("session_id")
    .distinct()
)

session_identity_checks = (
    sessions
    .join(
        duplicate_ids.withColumn("is_duplicate", lit(True)),
        on="session_id",
        how="left"
    )
    .withColumn(
        "DQ_SES_001_check",
        when(
            col("session_id").isNull()
            | (trim(col("session_id")) == "")
            | col("is_duplicate").isNotNull(),
            "FAIL"
        ).otherwise("PASS")
    )
    .drop("is_duplicate")
)

session_identity_checks.groupBy(
    "DQ_SES_001_check"
).count().show()

+----------------+------+
|DQ_SES_001_check| count|
+----------------+------+
|            FAIL|  1200|
|            PASS|298800|
+----------------+------+



In [0]:
session_lifecycle_checks = (
    session_chronology_checks
    .withColumn(
        "DQ_SES_004_check",
        when(
            # COMPLETED must have valid charging/end evidence
            (
                (col("final_status") == "COMPLETED")
                & (
                    col("charge_start_ts").isNull()
                    | col("charge_end_ts").isNull()
                    | col("departure_ts").isNull()
                    | col("end_reason").isNull()
                    | (trim(col("end_reason")) == "")
                    | col("energy_kwh").isNull()
                )
            )
            |
            # INTERRUPTED / CANCELLED must have departure and reason
            (
                col("final_status").isin("INTERRUPTED", "CANCELLED")
                & (
                    col("departure_ts").isNull()
                    | col("end_reason").isNull()
                    | (trim(col("end_reason")) == "")
                )
            ),
            "FAIL"
        ).otherwise("PASS")
    )
)

session_lifecycle_checks.groupBy(
    "DQ_SES_004_check"
).count().show()

+----------------+------+
|DQ_SES_004_check| count|
+----------------+------+
|            PASS|299798|
|            FAIL|   202|
+----------------+------+



In [0]:
session_lifecycle_checks.filter(
    col("DQ_SES_004_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "final_status",
    "arrival_ts",
    "charge_start_ts",
    "charge_end_ts",
    "departure_ts",
    "end_reason",
    "energy_kwh"
).show(30, truncate=False)

+------------------+------------+------------+-----------------------+-----------------------+-------------+-----------------------+---------------+----------+
|physical_record_id|session_id  |final_status|arrival_ts             |charge_start_ts        |charge_end_ts|departure_ts           |end_reason     |energy_kwh|
+------------------+------------+------------+-----------------------+-----------------------+-------------+-----------------------+---------------+----------+
|SESREC000000227   |SES000000227|COMPLETED   |2026-03-29 20:48:39.149|2026-03-29 20:50:39.149|NULL         |2026-03-29 21:47:39.149|TARGET_REACHED |11.615    |
|SESREC000000375   |SES000000375|COMPLETED   |2026-02-15 17:10:21.97 |2026-02-15 17:12:21.97 |NULL         |2026-02-15 17:34:21.97 |POWER_LIMIT    |23.225    |
|SESREC000000462   |SES000000462|COMPLETED   |2026-03-15 11:45:06.859|2026-03-15 11:48:06.859|NULL         |2026-03-15 12:03:06.859|VEHICLE_REQUEST|20.327    |
|SESREC000000843   |SES000000843|COMPLET

In [0]:
from pyspark.sql.functions import unix_timestamp

session_measures = (
    session_dq005_checks
    .withColumn(
        "duration_minutes",
        (
            unix_timestamp("charge_end_ts")
            - unix_timestamp("charge_start_ts")
        ) / 60.0
    )
    .withColumn(
        "occupied_minutes",
        (
            unix_timestamp("departure_ts")
            - unix_timestamp("arrival_ts")
        ) / 60.0
    )
)

In [0]:
session_measures = (
    session_measures
    .withColumn(
        "DQ_SES_006_check",
        when(
            col("energy_kwh").isNull()
            | (col("energy_kwh") < 0)
            | col("duration_minutes").isNull()
            | (col("duration_minutes") < 0)
            | col("occupied_minutes").isNull()
            | (col("occupied_minutes") < 0),
            "FAIL"
        ).otherwise("PASS")
    )
)

session_measures.groupBy(
    "DQ_SES_006_check"
).count().show()

+----------------+------+
|DQ_SES_006_check| count|
+----------------+------+
|            PASS|289945|
|            FAIL| 10055|
+----------------+------+



In [0]:
session_measures.filter(
    col("DQ_SES_006_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "energy_kwh",
    "duration_minutes",
    "occupied_minutes"
).show(30, truncate=False)

+------------------+------------+----------+----------------+----------------+
|physical_record_id|session_id  |energy_kwh|duration_minutes|occupied_minutes|
+------------------+------------+----------+----------------+----------------+
|SESREC000000014   |SES000000014|0.0       |NULL            |24.0            |
|SESREC000000020   |SES000000020|0.0       |NULL            |23.0            |
|SESREC000000068   |SES000000068|36.914    |130.0           |-5.0            |
|SESREC000000071   |SES000000071|0.0       |NULL            |10.0            |
|SESREC000000096   |SES000000096|0.0       |NULL            |18.0            |
|SESREC000000110   |SES000000110|0.0       |NULL            |9.0             |
|SESREC000000133   |SES000000133|0.0       |NULL            |16.0            |
|SESREC000000168   |SES000000168|0.0       |NULL            |18.0            |
|SESREC000000172   |SES000000172|0.0       |NULL            |24.0            |
|SESREC000000218   |SES000000218|0.0       |NULL    

In [0]:
from pyspark.sql.functions import unix_timestamp, col, when

session_measures = (
    session_dq005_checks
    .withColumn(
        "duration_minutes",
        (
            unix_timestamp("charge_end_ts")
            - unix_timestamp("charge_start_ts")
        ) / 60.0
    )
    .withColumn(
        "occupied_minutes",
        (
            unix_timestamp("departure_ts")
            - unix_timestamp("arrival_ts")
        ) / 60.0
    )
    .withColumn(
        "DQ_SES_006_check",
        when(
            col("energy_kwh").isNull()
            | (col("energy_kwh") < 0)
            | col("duration_minutes").isNull()
            | (col("duration_minutes") < 0)
            | col("occupied_minutes").isNull()
            | (col("occupied_minutes") < 0),
            "FAIL"
        ).otherwise("PASS")
    )
)

session_measures.groupBy(
    "DQ_SES_006_check"
).count().show()

+----------------+------+
|DQ_SES_006_check| count|
+----------------+------+
|            PASS|289945|
|            FAIL| 10055|
+----------------+------+



In [0]:
session_measures.filter(
    col("DQ_SES_006_check") == "FAIL"
).select(
    "physical_record_id",
    "session_id",
    "energy_kwh",
    "duration_minutes",
    "occupied_minutes"
).show(30, truncate=False)

+------------------+------------+----------+----------------+----------------+
|physical_record_id|session_id  |energy_kwh|duration_minutes|occupied_minutes|
+------------------+------------+----------+----------------+----------------+
|SESREC000000014   |SES000000014|0.0       |NULL            |24.0            |
|SESREC000000020   |SES000000020|0.0       |NULL            |23.0            |
|SESREC000000068   |SES000000068|36.914    |130.0           |-5.0            |
|SESREC000000071   |SES000000071|0.0       |NULL            |10.0            |
|SESREC000000096   |SES000000096|0.0       |NULL            |18.0            |
|SESREC000000110   |SES000000110|0.0       |NULL            |9.0             |
|SESREC000000133   |SES000000133|0.0       |NULL            |16.0            |
|SESREC000000168   |SES000000168|0.0       |NULL            |18.0            |
|SESREC000000172   |SES000000172|0.0       |NULL            |24.0            |
|SESREC000000218   |SES000000218|0.0       |NULL    

In [0]:
session_power = (
    session_measures
    .join(
        trusted_chargers_df.select(
            "charger_id",
            "rated_power_kw"
        ).distinct(),
        on="charger_id",
        how="left"
    )
    .withColumn(
        "expected_energy_kwh",
        col("rated_power_kw") * col("duration_minutes") / 60.0
    )
    .withColumn(
        "energy_ratio",
        when(
            col("expected_energy_kwh") > 0,
            col("energy_kwh") / col("expected_energy_kwh")
        )
    )
)

In [0]:
session_power.select(
    "energy_kwh",
    "rated_power_kw",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).filter(
    col("energy_ratio").isNotNull()
).summary().show()

+-------+------------------+-----------------+-----------------+-------------------+-------------------+
|summary|        energy_kwh|   rated_power_kw| duration_minutes|expected_energy_kwh|       energy_ratio|
+-------+------------------+-----------------+-----------------+-------------------+-------------------+
|  count|            290187|           290187|           290187|             290187|             290187|
|   mean| 25.44854103043689|50.31253088529644|73.43724908421122| 44.047324104801504| 0.6580298544851759|
| stddev|19.539243948911928|43.89713683382583|64.51258480803028| 60.289864083924705|0.31580534475080657|
|    min|              -3.5|              3.3|              3.0|0.16499999999999998| -7.070707070707071|
|    25%|             9.323|             22.0|             39.0|               20.0|              0.576|
|    50%|              20.2|             30.0|             61.0| 35.833333333333336|  0.740025974025974|
|    75%|            40.488|             60.0|         

In [0]:
session_power.select(
    "physical_record_id",
    "session_id",
    "ses.charger_id",
    "rated_power_kw",
    "energy_kwh",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).filter(
    col("energy_ratio").isNotNull()
).orderBy(
    col("energy_ratio").desc()
).show(20, truncate=False)

+------------------+------------+----------+--------------+----------+----------------+-------------------+------------------+
|physical_record_id|session_id  |charger_id|rated_power_kw|energy_kwh|duration_minutes|expected_energy_kwh|energy_ratio      |
+------------------+------------+----------+--------------+----------+----------------+-------------------+------------------+
|SESREC000008181   |SES000008181|CHG00032  |3.3           |7.229     |3.0             |0.16499999999999998|43.81212121212122 |
|SESREC000186668   |SES000186668|CHG00716  |60.0          |48.872    |3.0             |3.0                |16.290666666666667|
|SESREC000205776   |SES000205776|CHG00788  |3.3           |50.0      |65.0            |3.575              |13.986013986013985|
|SESREC000214918   |SES000214918|CHG00822  |7.4           |50.0      |31.0            |3.8233333333333333 |13.077593722755013|
|SESREC000270177   |SES000270177|CHG01037  |3.3           |50.0      |71.0            |3.905              |12.8

In [0]:
session_power.filter(
    col("energy_ratio").isNotNull()
).select(
    "physical_record_id",
    "session_id",
    "ses.charger_id",
    "rated_power_kw",
    "energy_kwh",
    "duration_minutes",
    "expected_energy_kwh",
    "energy_ratio"
).orderBy(
    col("energy_ratio").asc()
).show(20, truncate=False)

+------------------+------------+----------+--------------+----------+----------------+-------------------+--------------------+
|physical_record_id|session_id  |charger_id|rated_power_kw|energy_kwh|duration_minutes|expected_energy_kwh|energy_ratio        |
+------------------+------------+----------+--------------+----------+----------------+-------------------+--------------------+
|SESREC000011870   |SES000011870|CHG00046  |3.3           |-3.5      |9.0             |0.495              |-7.070707070707071  |
|SESREC000126771   |SES000126771|CHG00489  |7.4           |-3.5      |13.0            |1.6033333333333333 |-2.182952182952183  |
|SESREC000299428   |SES000299428|CHG01149  |7.4           |-3.5      |23.0            |2.836666666666667  |-1.2338425381903642 |
|SESREC000067254   |SES000067254|CHG00258  |3.3           |-3.5      |82.0            |4.51               |-0.7760532150776054 |
|SESREC000147018   |SES000147018|CHG00566  |22.0          |-3.5      |13.0            |4.76666666

In [0]:
session_all_rules = (
    session_power
    .withColumn(
        "DQ_SES_007_check",
        lit("PENDING_APPROVED_CONFIG")
    )
)

session_all_rules.select(
    "DQ_SES_007_check"
).groupBy(
    "DQ_SES_007_check"
).count().show()

+--------------------+------+
|    DQ_SES_007_check| count|
+--------------------+------+
|PENDING_APPROVED_...|300000|
+--------------------+------+



In [0]:
# Check for any configuration tables that may contain
# the approved DQ-SES-007 tolerance / efficiency values

spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

spark.sql("SHOW TABLES IN workspace.silver_layer").show(truncate=False)

+--------+--------------------------------------+-----------+
|database|tableName                             |isTemporary|
+--------+--------------------------------------+-----------+
|default |bronze_chargers                       |false      |
|default |bronze_maintenance                    |false      |
|default |bronze_sessions                       |false      |
|default |bronze_stations                       |false      |
|default |quarantine_chargeiq_chargers          |false      |
|default |quarantine_chargeiq_maintenance       |false      |
|default |quarantine_chargeiq_sessions          |false      |
|default |quarantine_chargeiq_stations          |false      |
|default |shiptrack_week03_bronze_demo_shipments|false      |
|default |shiptrack_week03_lineage_demo_view    |false      |
|default |trusted_silver_chargeiq_chargers      |false      |
|default |trusted_silver_chargeiq_maintenance   |false      |
|default |trusted_silver_chargeiq_sessions      |false      |
|default

In [0]:
# Search for tables/views related to DQ, config, tolerance, efficiency, or rules

for schema in ["workspace.default", "workspace.silver_layer"]:
    print(f"\n===== {schema} =====")
    spark.sql(f"SHOW TABLES IN {schema}").filter(
        """
        lower(tableName) like '%dq%'
        or lower(tableName) like '%config%'
        or lower(tableName) like '%rule%'
        or lower(tableName) like '%tolerance%'
        or lower(tableName) like '%efficien%'
        """
    ).show(truncate=False)


===== workspace.default =====
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+


===== workspace.silver_layer =====
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [0]:
# Search the Week 6 notebook source for DQ-SES-007 configuration terms

import re

notebook_path = "/Workspace/Users/<your-username>/04_data_quality_checks"

# If the path above is not your actual notebook path,
# use the notebook's existing location in the Workspace.

In [0]:
print("DQ-SES-007 STATUS: BLOCKED")
print("Reason: Approved efficiency/tolerance configuration is not available")
print("Action: Do not invent threshold; obtain approved project configuration")

DQ-SES-007 STATUS: BLOCKED
Reason: Approved efficiency/tolerance configuration is not available
Action: Do not invent threshold; obtain approved project configuration


In [0]:
from pyspark.sql import functions as F

session_all_checks = (
    session_measures
    .withColumn(
        "failed_rule_ids",
        F.concat_ws(
            ",",
            F.when(F.col("DQ_SES_001_check") == "FAIL", F.lit("DQ-SES-001")),
            F.when(F.col("DQ_SES_002_check") == "FAIL", F.lit("DQ-SES-002")),
            F.when(F.col("DQ_SES_003_check") == "FAIL", F.lit("DQ-SES-003")),
            F.when(F.col("DQ_SES_004_check") == "FAIL", F.lit("DQ-SES-004")),
            F.when(F.col("DQ_SES_005_check") == "FAIL", F.lit("DQ-SES-005")),
            F.when(F.col("DQ_SES_006_check") == "FAIL", F.lit("DQ-SES-006"))
        )
    )
    .withColumn(
        "dq_route",
        F.when(
            F.col("failed_rule_ids") != "",
            F.lit("QUARANTINE")
        ).otherwise(F.lit("TRUSTED"))
    )
)

session_all_checks.groupBy("dq_route").count().show()

+----------+------+
|  dq_route| count|
+----------+------+
|   TRUSTED|286902|
|QUARANTINE| 13098|
+----------+------+



In [0]:
TRUSTED_SESSIONS = "workspace.silver_layer.trusted_sessions"
QUAR_SESSIONS = "workspace.silver_layer.quarantine_sessions"

trusted_sessions_df = (
    session_all_checks
    .filter(F.col("dq_route") == "TRUSTED")
    .select(
        "physical_record_id",
        "session_id",
        "ses.station_id",
        "ses.charger_id",
        "vehicle_class",
        "connector_type",
        "arrival_ts",
        "charge_start_ts",
        "charge_end_ts",
        "departure_ts",
        "final_status",
        "end_reason",
        "energy_kwh",
        "start_soc_pct",
        "end_soc_pct",
        "tariff_band",
        "tariff_rate_inr_per_kwh",
        "meter_quality_flag",
        "batch_date",
        "source_system",
        "ingestion_time",
        "source_file",
        "run_id",
        "_candidate_created_at",
        "_candidate_schema_version",
        "duration_minutes",
        "occupied_minutes"
    )
)

trusted_sessions_df.write.mode("overwrite").saveAsTable(TRUSTED_SESSIONS)

In [0]:
quarantine_sessions_df = (
    session_all_checks
    .filter(F.col("dq_route") == "QUARANTINE")
    .select(
        "physical_record_id",
        "session_id",
        "ses.station_id",
        "ses.charger_id",
        "vehicle_class",
        "connector_type",
        "arrival_ts",
        "charge_start_ts",
        "charge_end_ts",
        "departure_ts",
        "final_status",
        "end_reason",
        "energy_kwh",
        "start_soc_pct",
        "end_soc_pct",
        "tariff_band",
        "tariff_rate_inr_per_kwh",
        "meter_quality_flag",
        "batch_date",
        "source_system",
        "ingestion_time",
        "source_file",
        "run_id",
        "_candidate_created_at",
        "_candidate_schema_version",
        "DQ_SES_001_check",
        "DQ_SES_002_check",
        "DQ_SES_003_check",
        "DQ_SES_004_check",
        "DQ_SES_005_overlap",
        "DQ_SES_005_capacity",
        "DQ_SES_005_check",
        "DQ_SES_005_reason",
        "DQ_SES_006_check",
        "duration_minutes",
        "occupied_minutes",
        "failed_rule_ids",
        "dq_route"
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "rework_status",
        F.lit("PENDING")
    )
)

quarantine_sessions_df.write.mode("overwrite").saveAsTable(QUAR_SESSIONS)

In [0]:
print("Candidate:")
spark.table("workspace.silver_layer.silver_sessions_candidate") \
    .select("physical_record_id").distinct().count()

print("Trusted:")
spark.table(TRUSTED_SESSIONS) \
    .select("physical_record_id").distinct().count()

print("Quarantine:")
spark.table(QUAR_SESSIONS) \
    .select("physical_record_id").distinct().count()

Candidate:
Trusted:
Quarantine:


13098

In [0]:
print("Candidate :", spark.table(
    "workspace.silver_layer.silver_sessions_candidate"
).select("physical_record_id").distinct().count())

print("Trusted   :", spark.table(
    "workspace.silver_layer.trusted_sessions"
).select("physical_record_id").distinct().count())

print("Quarantine:", spark.table(
    "workspace.silver_layer.quarantine_sessions"
).select("physical_record_id").distinct().count())

Candidate : 300000
Trusted   : 286902
Quarantine: 13098


In [0]:
trusted_ids = (
    spark.table("workspace.silver_layer.trusted_sessions")
    .select("physical_record_id")
    .distinct()
)

quarantine_ids = (
    spark.table("workspace.silver_layer.quarantine_sessions")
    .select("physical_record_id")
    .distinct()
)

overlap = (
    trusted_ids
    .join(quarantine_ids, "physical_record_id", "inner")
)

print("Overlap:", overlap.count())

Overlap: 0


In [0]:
spark.table("workspace.silver_layer.quarantine_sessions").select(
    "physical_record_id",
    "session_id",
    "station_id",
    "charger_id",
    "failed_rule_ids",
    "DQ_SES_001_check",
    "DQ_SES_002_check",
    "DQ_SES_003_check",
    "DQ_SES_004_check",
    "DQ_SES_005_check",
    "DQ_SES_006_check",
    "quarantined_at",
    "rework_status"
).show(20, truncate=False)

+------------------+------------+----------+----------+---------------------+----------------+----------------+----------------+----------------+----------------+----------------+--------------------------+-------------+
|physical_record_id|session_id  |station_id|charger_id|failed_rule_ids      |DQ_SES_001_check|DQ_SES_002_check|DQ_SES_003_check|DQ_SES_004_check|DQ_SES_005_check|DQ_SES_006_check|quarantined_at            |rework_status|
+------------------+------------+----------+----------+---------------------+----------------+----------------+----------------+----------------+----------------+----------------+--------------------------+-------------+
|SESREC000000268   |SES000000268|STN0001   |CHG00002  |DQ-SES-003,DQ-SES-006|PASS            |PASS            |FAIL            |PASS            |PASS            |FAIL            |2026-09-10 11:39:52.057354|PENDING      |
|SESREC000002927   |SES000002927|STN0002   |CHG00012  |DQ-SES-003,DQ-SES-006|PASS            |PASS            |FAIL 

In [0]:
from pyspark.sql import functions as F

quar = spark.table("workspace.silver_layer.quarantine_sessions")

print("Quarantine rows:", quar.count())
print("Distinct physical records:", quar.select("physical_record_id").distinct().count())

quar.select(
    "physical_record_id",
    "session_id",
    "failed_rule_ids",
    "quarantined_at",
    "rework_status"
).show(20, truncate=False)

Quarantine rows: 13098
Distinct physical records: 13098
+------------------+------------+---------------------+--------------------------+-------------+
|physical_record_id|session_id  |failed_rule_ids      |quarantined_at            |rework_status|
+------------------+------------+---------------------+--------------------------+-------------+
|SESREC000000268   |SES000000268|DQ-SES-003,DQ-SES-006|2026-09-10 11:39:52.057354|PENDING      |
|SESREC000002927   |SES000002927|DQ-SES-003,DQ-SES-006|2026-09-10 11:39:52.057354|PENDING      |
|SESREC000003280   |SES000003280|DQ-SES-002           |2026-09-10 11:39:52.057354|PENDING      |
|SESREC000003896   |SES000003896|DQ-SES-003,DQ-SES-006|2026-09-10 11:39:52.057354|PENDING      |
|SESREC000009122   |SES000009122|DQ-SES-005           |2026-09-10 11:39:52.057354|PENDING      |
|SESREC000010071   |SES000010071|DQ-SES-003,DQ-SES-006|2026-09-10 11:39:52.057354|PENDING      |
|SESREC000010524   |SES000010524|DQ-SES-006           |2026-09-10 11:39

In [0]:
quar.groupBy("failed_rule_ids") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+--------------------------------+-----+
|failed_rule_ids                 |count|
+--------------------------------+-----+
|DQ-SES-003,DQ-SES-006           |9534 |
|DQ-SES-005                      |1535 |
|DQ-SES-002                      |822  |
|DQ-SES-001                      |391  |
|DQ-SES-006                      |288  |
|DQ-SES-003                      |238  |
|DQ-SES-003,DQ-SES-004,DQ-SES-006|202  |
|DQ-SES-002,DQ-SES-005           |56   |
|DQ-SES-002,DQ-SES-003,DQ-SES-006|22   |
|DQ-SES-001,DQ-SES-003,DQ-SES-006|8    |
|DQ-SES-005,DQ-SES-006           |1    |
|DQ-SES-001,DQ-SES-005           |1    |
+--------------------------------+-----+



In [0]:
rule_occurrences = (
    quar
    .select(
        "physical_record_id",
        F.explode(
            F.split("failed_rule_ids", ",")
        ).alias("rule_id")
    )
)

rule_occurrences.groupBy("rule_id") \
    .count() \
    .orderBy("rule_id") \
    .show()

+----------+-----+
|   rule_id|count|
+----------+-----+
|DQ-SES-001|  400|
|DQ-SES-002|  900|
|DQ-SES-003|10004|
|DQ-SES-004|  202|
|DQ-SES-005| 1593|
|DQ-SES-006|10055|
+----------+-----+



In [0]:
print("Physical quarantine records:",
      quar.select("physical_record_id").distinct().count())

print("Rule occurrences:",
      rule_occurrences.count())

Physical quarantine records: 13098
Rule occurrences: 23154


In [0]:
from pyspark.sql import functions as F

dq_rule_occurrences = (
    quar
    .select(
        "physical_record_id",
        "session_id",
        "station_id",
        "charger_id",
        "failed_rule_ids",
        "source_system",
        "source_file",
        "run_id",
        "quarantined_at",
        "rework_status"
    )
    .withColumn(
        "rule_id",
        F.explode(F.split("failed_rule_ids", ","))
    )
    .withColumn(
        "rule_name",
        F.when(F.col("rule_id") == "DQ-SES-001", "Session identity")
         .when(F.col("rule_id") == "DQ-SES-002", "Station/charger reference and alignment")
         .when(F.col("rule_id") == "DQ-SES-003", "Chronology/window")
         .when(F.col("rule_id") == "DQ-SES-004", "Lifecycle")
         .when(F.col("rule_id") == "DQ-SES-005", "Occupancy/capacity")
         .when(F.col("rule_id") == "DQ-SES-006", "Measures/ranges")
    )
    .withColumn(
        "severity",
        F.when(
            F.col("rule_id").isin(
                "DQ-SES-001",
                "DQ-SES-002",
                "DQ-SES-005"
            ),
            "CRITICAL"
        ).otherwise("MAJOR")
    )
)

dq_rule_occurrences.groupBy(
    "rule_id",
    "rule_name",
    "severity"
).count().orderBy("rule_id").show(truncate=False)

+----------+---------------------------------------+--------+-----+
|rule_id   |rule_name                              |severity|count|
+----------+---------------------------------------+--------+-----+
|DQ-SES-001|Session identity                       |CRITICAL|400  |
|DQ-SES-002|Station/charger reference and alignment|CRITICAL|900  |
|DQ-SES-003|Chronology/window                      |MAJOR   |10004|
|DQ-SES-004|Lifecycle                              |MAJOR   |202  |
|DQ-SES-005|Occupancy/capacity                     |CRITICAL|1593 |
|DQ-SES-006|Measures/ranges                        |MAJOR   |10055|
+----------+---------------------------------------+--------+-----+



In [0]:
DQ_RULE_OCCURRENCES = "workspace.silver_layer.dq_session_rule_occurrences"

dq_rule_occurrences.write \
    .mode("overwrite") \
    .saveAsTable(DQ_RULE_OCCURRENCES)

In [0]:
print(
    "Rule occurrence records:",
    spark.table(DQ_RULE_OCCURRENCES).count()
)

print(
    "Distinct quarantined physical records:",
    spark.table(DQ_RULE_OCCURRENCES)
    .select("physical_record_id")
    .distinct()
    .count()
)

Rule occurrence records: 23154
Distinct quarantined physical records: 13098


In [0]:
DQ_RULE_OCCURRENCES = "workspace.silver_layer.dq_session_rule_occurrences"

dq_rule_occurrences.write \
    .mode("overwrite") \
    .saveAsTable(DQ_RULE_OCCURRENCES)

In [0]:
print(
    "Rule occurrence records:",
    spark.table(DQ_RULE_OCCURRENCES).count()
)

print(
    "Distinct physical records:",
    spark.table(DQ_RULE_OCCURRENCES)
    .select("physical_record_id")
    .distinct()
    .count()
)

spark.table(DQ_RULE_OCCURRENCES) \
    .groupBy("rule_id", "severity") \
    .count() \
    .orderBy("rule_id") \
    .show()

Rule occurrence records: 23154
Distinct physical records: 13098
+----------+--------+-----+
|   rule_id|severity|count|
+----------+--------+-----+
|DQ-SES-001|CRITICAL|  400|
|DQ-SES-002|CRITICAL|  900|
|DQ-SES-003|   MAJOR|10004|
|DQ-SES-004|   MAJOR|  202|
|DQ-SES-005|CRITICAL| 1593|
|DQ-SES-006|   MAJOR|10055|
+----------+--------+-----+



In [0]:
from pyspark.sql import functions as F

maintenance = spark.table(
    "workspace.silver_layer.silver_maintenance_candidate"
)

print("Maintenance candidate rows:", maintenance.count())

maintenance.printSchema()

Maintenance candidate rows: 18000
root
 |-- physical_record_id: string (nullable = true)
 |-- maintenance_id: string (nullable = true)
 |-- incident_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- fault_category: string (nullable = true)
 |-- fault_code: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- status_after: string (nullable = true)
 |-- related_fault_event_id: string (nullable = true)
 |-- planned_flag: boolean (nullable = true)
 |-- notes_code: string (nullable = true)
 |-- batch_date: date (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- _candidate_created_at: timestamp (nullable = true)
 |-- _candidate_schema_version: string (nullable = true)



In [0]:
maintenance.show(10, truncate=False)

+------------------+--------------+-----------+----------+----------+-------------------+------------------+--------------+----------+--------+------------+----------------------+------------+------------------------------------+----------+--------------------+--------------------------+-------------------------------------------------------+-------+--------------------------+-------------------------+
|physical_record_id|maintenance_id|incident_id|station_id|charger_id|event_ts           |event_type        |fault_category|fault_code|severity|status_after|related_fault_event_id|planned_flag|notes_code                          |batch_date|source_system       |ingestion_time            |source_file                                            |run_id |_candidate_created_at     |_candidate_schema_version|
+------------------+--------------+-----------+----------+----------+-------------------+------------------+--------------+----------+--------+------------+----------------------+---------

In [0]:
from pyspark.sql import functions as F

# Trusted reference data
station_ref = (
    trusted_stations_df
    .select("station_id")
    .distinct()
)

charger_ref = (
    trusted_chargers_df
    .select("charger_id", "station_id")
    .distinct()
)

# Add trusted-reference checks
maintenance_ref_checks = (
    maintenance.alias("m")
    .join(
        station_ref.alias("s"),
        F.col("m.station_id") == F.col("s.station_id"),
        "left"
    )
    .join(
        charger_ref.alias("c"),
        F.col("m.charger_id") == F.col("c.charger_id"),
        "left"
    )
    .withColumn(
        "station_resolved",
        F.col("s.station_id").isNotNull()
    )
    .withColumn(
        "charger_resolved",
        F.col("c.charger_id").isNotNull()
    )
    .withColumn(
        "charger_station_aligned",
        F.col("c.station_id") == F.col("m.station_id")
    )
)

In [0]:
fault_events = (
    maintenance
    .filter(F.col("event_type") == "FAULT_REPORTED")
    .select(
        F.col("incident_id").alias("fault_incident_id"),
        F.col("maintenance_id").alias("fault_maintenance_id"),
        F.col("station_id").alias("fault_station_id"),
        F.col("charger_id").alias("fault_charger_id"),
        F.col("event_ts").alias("fault_ts")
    )
)

maintenance_checks = (
    maintenance_ref_checks
    .join(
        fault_events,
        F.col("incident_id") == F.col("fault_incident_id"),
        "left"
    )
    .withColumn(
        "DQ_MNT_001_check",
        F.when(
            # Missing/unresolved references
            F.col("m.station_id").isNull()
            | (F.trim(F.col("m.station_id")) == "")
            | F.col("m.charger_id").isNull()
            | (F.trim(F.col("m.charger_id")) == "")
            | ~F.col("station_resolved")
            | ~F.col("charger_resolved")
            | ~F.col("charger_station_aligned")
            |
            # Missing event identity/time
            F.col("m.incident_id").isNull()
            | (F.trim(F.col("m.incident_id")) == "")
            | F.col("m.event_ts").isNull()
            |
            # Repair/recovery must link to a known fault
            (
                F.col("m.event_type").isin(
                    "REPAIR_STARTED",
                    "RECOVERY_CONFIRMED"
                )
                & F.col("fault_maintenance_id").isNull()
            )
            |
            # Repair must occur after fault
            (
                (F.col("m.event_type") == "REPAIR_STARTED")
                & (F.col("m.event_ts") <= F.col("fault_ts"))
            )
            |
            # Recovery must occur after fault
            (
                (F.col("m.event_type") == "RECOVERY_CONFIRMED")
                & (F.col("m.event_ts") <= F.col("fault_ts"))
            ),
            "FAIL"
        ).otherwise("PASS")
    )
)

In [0]:
maintenance_checks.groupBy("DQ_MNT_001_check").count().show()

+----------------+-----+
|DQ_MNT_001_check|count|
+----------------+-----+
|            PASS|17849|
|            FAIL|  151|
+----------------+-----+



In [0]:
maintenance_checks \
    .filter(F.col("DQ_MNT_001_check") == "FAIL") \
    .select(
        "physical_record_id",
        "maintenance_id",
        "incident_id",
        "m.station_id",
        "m.charger_id",
        "event_ts",
        "event_type",
        "related_fault_event_id",
        "fault_maintenance_id",
        "fault_ts"
    ) \
    .show(20, truncate=False)

+------------------+--------------+-----------+----------+----------+-------------------+------------------+----------------------+--------------------+-------------------+
|physical_record_id|maintenance_id|incident_id|station_id|charger_id|event_ts           |event_type        |related_fault_event_id|fault_maintenance_id|fault_ts           |
+------------------+--------------+-----------+----------+----------+-------------------+------------------+----------------------+--------------------+-------------------+
|MNTREC00000023    |MNT00000023   |INC000008  |STN0083   |CHG99999  |2026-03-30 11:04:00|REPAIR_STARTED    |MNT00000022           |MNT00000022         |2026-03-30 09:40:00|
|MNTREC00000063    |MNT00000063   |INC000021  |STN0001   |CHG00885  |2026-01-28 02:21:00|RECOVERY_CONFIRMED|MNT00000061           |MNT00000061         |2026-01-27 21:48:00|
|MNTREC00000108    |MNT00000108   |INC000036  |STN0151   |CHG01005  |2025-12-22 00:00:00|RECOVERY_CONFIRMED|MNT00000106           |MNT0

In [0]:
maintenance_checks = (
    maintenance_checks
    .withColumn(
        "DQ_MNT_001_reason",
        F.concat_ws(
            "; ",
            F.when(
                ~F.col("station_resolved"),
                F.lit("Station unresolved")
            ),
            F.when(
                ~F.col("charger_resolved"),
                F.lit("Charger unresolved")
            ),
            F.when(
                ~F.col("charger_station_aligned"),
                F.lit("Charger does not belong to session station")
            ),
            F.when(
                F.col("event_ts").isNull(),
                F.lit("Missing event timestamp")
            ),
            F.when(
                (
                    F.col("event_type").isin(
                        "REPAIR_STARTED",
                        "RECOVERY_CONFIRMED"
                    )
                )
                & F.col("fault_maintenance_id").isNull(),
                F.lit("No linked fault event")
            ),
            F.when(
                (
                    F.col("event_type").isin(
                        "REPAIR_STARTED",
                        "RECOVERY_CONFIRMED"
                    )
                )
                & (F.col("event_ts") <= F.col("fault_ts")),
                F.lit("Event timestamp not after fault")
            )
        )
    )
)

maintenance_checks.groupBy(
    "DQ_MNT_001_check",
    "DQ_MNT_001_reason"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+----------------+------------------------------------------+-----+
|DQ_MNT_001_check|DQ_MNT_001_reason                         |count|
+----------------+------------------------------------------+-----+
|PASS            |                                          |17849|
|FAIL            |Charger unresolved                        |60   |
|FAIL            |Charger does not belong to session station|60   |
|FAIL            |Event timestamp not after fault           |31   |
+----------------+------------------------------------------+-----+



In [0]:
maintenance_checks \
    .filter(F.col("DQ_MNT_001_check") == "FAIL") \
    .groupBy("event_type", "DQ_MNT_001_reason") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

+------------------+------------------------------------------+-----+
|event_type        |DQ_MNT_001_reason                         |count|
+------------------+------------------------------------------+-----+
|INSPECTION        |Charger does not belong to session station|17   |
|INSPECTION        |Charger unresolved                        |17   |
|REPAIR_STARTED    |Charger unresolved                        |16   |
|REPAIR_STARTED    |Event timestamp not after fault           |16   |
|REPAIR_STARTED    |Charger does not belong to session station|16   |
|RECOVERY_CONFIRMED|Event timestamp not after fault           |15   |
|RECOVERY_CONFIRMED|Charger unresolved                        |14   |
|FAULT_REPORTED    |Charger does not belong to session station|14   |
|FAULT_REPORTED    |Charger unresolved                        |13   |
|RECOVERY_CONFIRMED|Charger does not belong to session station|13   |
+------------------+------------------------------------------+-----+



In [0]:
maintenance_final = (
    maintenance_checks
    .withColumn(
        "failed_rule_ids",
        F.when(
            F.col("DQ_MNT_001_check") == "FAIL",
            F.lit("DQ-MNT-001")
        ).otherwise(F.lit(""))
    )
    .withColumn(
        "dq_route",
        F.when(
            F.col("DQ_MNT_001_check") == "FAIL",
            F.lit("QUARANTINE")
        ).otherwise(F.lit("TRUSTED"))
    )
)

In [0]:
maintenance_final.groupBy("dq_route").count().show()

+----------+-----+
|  dq_route|count|
+----------+-----+
|QUARANTINE|  151|
|   TRUSTED|17849|
+----------+-----+



In [0]:
TRUSTED_MAINTENANCE = "workspace.silver_layer.trusted_maintenance"
QUAR_MAINTENANCE = "workspace.silver_layer.quarantine_maintenance"

trusted_maintenance_df = (
    maintenance_final
    .filter(F.col("dq_route") == "TRUSTED")
    .select(
        "physical_record_id",
        "maintenance_id",
        "incident_id",
        "m.station_id",
        "m.charger_id",
        "event_ts",
        "event_type",
        "fault_category",
        "fault_code",
        "severity",
        "status_after",
        "related_fault_event_id",
        "planned_flag",
        "notes_code",
        "batch_date",
        "source_system",
        "ingestion_time",
        "source_file",
        "run_id",
        "_candidate_created_at",
        "_candidate_schema_version"
    )
)

trusted_maintenance_df.write \
    .mode("overwrite") \
    .saveAsTable(TRUSTED_MAINTENANCE)

In [0]:
quarantine_maintenance_df = (
    maintenance_final
    .filter(F.col("dq_route") == "QUARANTINE")
    .select(
        "physical_record_id",
        "maintenance_id",
        "incident_id",
        "m.station_id",
        "m.charger_id",
        "event_ts",
        "event_type",
        "fault_category",
        "fault_code",
        "severity",
        "status_after",
        "related_fault_event_id",
        "planned_flag",
        "notes_code",
        "batch_date",
        "source_system",
        "ingestion_time",
        "source_file",
        "run_id",
        "_candidate_created_at",
        "_candidate_schema_version",
        "DQ_MNT_001_check",
        "DQ_MNT_001_reason",
        "failed_rule_ids",
        "dq_route"
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "rework_status",
        F.lit("PENDING")
    )
)

quarantine_maintenance_df.write \
    .mode("overwrite") \
    .saveAsTable(QUAR_MAINTENANCE)

In [0]:
candidate_count = (
    maintenance
    .select("physical_record_id")
    .distinct()
    .count()
)

trusted_count = (
    spark.table(TRUSTED_MAINTENANCE)
    .select("physical_record_id")
    .distinct()
    .count()
)

quarantine_count = (
    spark.table(QUAR_MAINTENANCE)
    .select("physical_record_id")
    .distinct()
    .count()
)

overlap_count = (
    spark.table(TRUSTED_MAINTENANCE)
    .select("physical_record_id")
    .distinct()
    .join(
        spark.table(QUAR_MAINTENANCE)
        .select("physical_record_id")
        .distinct(),
        "physical_record_id",
        "inner"
    )
    .count()
)

print("Candidate :", candidate_count)
print("Trusted   :", trusted_count)
print("Quarantine:", quarantine_count)
print("Overlap   :", overlap_count)
print("Variance  :", candidate_count - trusted_count - quarantine_count)

Candidate : 18000
Trusted   : 17849
Quarantine: 151
Overlap   : 0
Variance  : 0


In [0]:
spark.sql("SHOW TABLES IN workspace.silver_layer").show(truncate=False)

+------------+----------------------------+-----------+
|database    |tableName                   |isTemporary|
+------------+----------------------------+-----------+
|silver_layer|dq_session_rule_occurrences |false      |
|silver_layer|quarantine_chargers         |false      |
|silver_layer|quarantine_maintenance      |false      |
|silver_layer|quarantine_sessions         |false      |
|silver_layer|quarantine_stations         |false      |
|silver_layer|silver_chargers_candidate   |false      |
|silver_layer|silver_maintenance_candidate|false      |
|silver_layer|silver_sessions_candidate   |false      |
|silver_layer|silver_stations_candidate   |false      |
|silver_layer|trusted_chargers            |false      |
|silver_layer|trusted_maintenance         |false      |
|silver_layer|trusted_sessions            |false      |
|silver_layer|trusted_stations            |false      |
+------------+----------------------------+-----------+



In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+--------------------------------------+-----------+
|database|tableName                             |isTemporary|
+--------+--------------------------------------+-----------+
|default |bronze_chargers                       |false      |
|default |bronze_maintenance                    |false      |
|default |bronze_sessions                       |false      |
|default |bronze_stations                       |false      |
|default |quarantine_chargeiq_chargers          |false      |
|default |quarantine_chargeiq_maintenance       |false      |
|default |quarantine_chargeiq_sessions          |false      |
|default |quarantine_chargeiq_stations          |false      |
|default |shiptrack_week03_bronze_demo_shipments|false      |
|default |shiptrack_week03_lineage_demo_view    |false      |
|default |trusted_silver_chargeiq_chargers      |false      |
|default |trusted_silver_chargeiq_maintenance   |false      |
|default |trusted_silver_chargeiq_sessions      |false      |
|default

In [0]:
from pyspark.sql import Row

week6_status = [
    Row(
        rule_id="DQ-EVT-001",
        rule_name="Status events streaming governance",
        severity="MAJOR",
        status="BLOCKED",
        reason="No charger status events Bronze/Candidate source exists in the Week 5 environment",
        action="Await upstream Events source/table before implementation"
    ),
    Row(
        rule_id="DQ-SES-007",
        rule_name="Physical plausibility",
        severity="MAJOR",
        status="BLOCKED",
        reason="Approved efficiency/tolerance configuration is not available",
        action="Obtain approved project configuration before applying rule"
    )
]

status_df = spark.createDataFrame(week6_status)

status_df.show(truncate=False)

+----------+----------------------------------+--------+-------+---------------------------------------------------------------------------------+----------------------------------------------------------+
|rule_id   |rule_name                         |severity|status |reason                                                                           |action                                                    |
+----------+----------------------------------+--------+-------+---------------------------------------------------------------------------------+----------------------------------------------------------+
|DQ-EVT-001|Status events streaming governance|MAJOR   |BLOCKED|No charger status events Bronze/Candidate source exists in the Week 5 environment|Await upstream Events source/table before implementation  |
|DQ-SES-007|Physical plausibility             |MAJOR   |BLOCKED|Approved efficiency/tolerance configuration is not available                     |Obtain approved project config

In [0]:
print("=== WEEK 6 OUTPUT COUNTS ===")

print(
    "Trusted Stations:",
    spark.table("workspace.silver_layer.trusted_stations")
    .select("physical_record_id").distinct().count()
)

print(
    "Quarantine Stations:",
    spark.table("workspace.silver_layer.quarantine_stations")
    .select("physical_record_id").distinct().count()
)

print(
    "Trusted Chargers:",
    spark.table("workspace.silver_layer.trusted_chargers")
    .select("physical_record_id").distinct().count()
)

print(
    "Quarantine Chargers:",
    spark.table("workspace.silver_layer.quarantine_chargers")
    .select("physical_record_id").distinct().count()
)

print(
    "Trusted Sessions:",
    spark.table("workspace.silver_layer.trusted_sessions")
    .select("physical_record_id").distinct().count()
)

print(
    "Quarantine Sessions:",
    spark.table("workspace.silver_layer.quarantine_sessions")
    .select("physical_record_id").distinct().count()
)

print(
    "Trusted Maintenance:",
    spark.table("workspace.silver_layer.trusted_maintenance")
    .select("physical_record_id").distinct().count()
)

print(
    "Quarantine Maintenance:",
    spark.table("workspace.silver_layer.quarantine_maintenance")
    .select("physical_record_id").distinct().count()
)

print(
    "Session DQ Rule Occurrences:",
    spark.table("workspace.silver_layer.dq_session_rule_occurrences").count()
)

=== WEEK 6 OUTPUT COUNTS ===
Trusted Stations: 178
Quarantine Stations: 2
Trusted Chargers: 1177
Quarantine Chargers: 23
Trusted Sessions: 286902
Quarantine Sessions: 13098
Trusted Maintenance: 17849
Quarantine Maintenance: 151
Session DQ Rule Occurrences: 23154


In [0]:
from pyspark.sql import Row

# ============================================================
# WEEK 6 FINAL RECONCILIATION
# ============================================================

tables = {
    "Stations Candidate": "workspace.silver_layer.silver_stations_candidate",
    "Stations Trusted": "workspace.silver_layer.trusted_stations",
    "Stations Quarantine": "workspace.silver_layer.quarantine_stations",

    "Chargers Candidate": "workspace.silver_layer.silver_chargers_candidate",
    "Chargers Trusted": "workspace.silver_layer.trusted_chargers",
    "Chargers Quarantine": "workspace.silver_layer.quarantine_chargers",

    "Sessions Candidate": "workspace.silver_layer.silver_sessions_candidate",
    "Sessions Trusted": "workspace.silver_layer.trusted_sessions",
    "Sessions Quarantine": "workspace.silver_layer.quarantine_sessions",

    "Maintenance Candidate": "workspace.silver_layer.silver_maintenance_candidate",
    "Maintenance Trusted": "workspace.silver_layer.trusted_maintenance",
    "Maintenance Quarantine": "workspace.silver_layer.quarantine_maintenance"
}

results = []

for name, table_name in tables.items():
    count = spark.table(table_name).select("physical_record_id").distinct().count()
    results.append(Row(table=name, distinct_physical_records=count))

recon_df = spark.createDataFrame(results)

display(
    recon_df.orderBy("table")
)

table,distinct_physical_records
Chargers Candidate,1200
Chargers Quarantine,23
Chargers Trusted,1177
Maintenance Candidate,18000
Maintenance Quarantine,151
Maintenance Trusted,17849
Sessions Candidate,300000
Sessions Quarantine,13098
Sessions Trusted,286902
Stations Candidate,180


In [0]:
domains = [
    ("stations",
     "workspace.silver_layer.trusted_stations",
     "workspace.silver_layer.quarantine_stations"),

    ("chargers",
     "workspace.silver_layer.trusted_chargers",
     "workspace.silver_layer.quarantine_chargers"),

    ("sessions",
     "workspace.silver_layer.trusted_sessions",
     "workspace.silver_layer.quarantine_sessions"),

    ("maintenance",
     "workspace.silver_layer.trusted_maintenance",
     "workspace.silver_layer.quarantine_maintenance")
]

overlap_results = []

for domain, trusted_table, quarantine_table in domains:

    trusted_ids = (
        spark.table(trusted_table)
        .select("physical_record_id")
        .distinct()
    )

    quarantine_ids = (
        spark.table(quarantine_table)
        .select("physical_record_id")
        .distinct()
    )

    overlap = trusted_ids.join(
        quarantine_ids,
        "physical_record_id",
        "inner"
    ).count()

    overlap_results.append(
        Row(
            domain=domain,
            trusted_count=trusted_ids.count(),
            quarantine_count=quarantine_ids.count(),
            overlap_count=overlap
        )
    )

overlap_df = spark.createDataFrame(overlap_results)

display(overlap_df)

domain,trusted_count,quarantine_count,overlap_count
stations,178,2,0
chargers,1177,23,0
sessions,286902,13098,0
maintenance,17849,151,0


In [0]:
week6_status = [
    Row(
        rule_id="DQ-CHG-002",
        domain="Stations",
        severity="CRITICAL",
        status="PASS",
        reason="Station Candidate records evaluated and routed.",
        action="None"
    ),

    Row(
        rule_id="DQ-CHG-001",
        domain="Chargers",
        severity="CRITICAL",
        status="PASS",
        reason="Charger Candidate records evaluated and routed.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-001",
        domain="Sessions",
        severity="CRITICAL",
        status="PASS",
        reason="Session identity and uniqueness checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-002",
        domain="Sessions",
        severity="CRITICAL",
        status="PASS",
        reason="Station/charger reference and alignment checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-003",
        domain="Sessions",
        severity="MAJOR",
        status="PASS",
        reason="Timestamp parsing, chronology and reporting-window checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-004",
        domain="Sessions",
        severity="MAJOR",
        status="PASS",
        reason="Lifecycle consistency checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-005",
        domain="Sessions",
        severity="CRITICAL",
        status="PASS",
        reason="Charger overlap and station capacity checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-006",
        domain="Sessions",
        severity="MAJOR",
        status="PASS",
        reason="Measure validity checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-SES-007",
        domain="Sessions",
        severity="MAJOR",
        status="BLOCKED",
        reason="Approved efficiency/tolerance configuration is not available in the Week 5/Week 6 environment.",
        action="Obtain approved project configuration before enabling this rule."
    ),

    Row(
        rule_id="DQ-MNT-001",
        domain="Maintenance",
        severity="MAJOR",
        status="PASS",
        reason="Maintenance reference integrity and chronology checks implemented.",
        action="None"
    ),

    Row(
        rule_id="DQ-EVT-001",
        domain="Status Events",
        severity="MAJOR",
        status="BLOCKED",
        reason="No charger status-events Bronze/Candidate source exists in the available Week 5 environment.",
        action="Obtain upstream Events source/table before implementing this rule."
    )
]

week6_status_df = spark.createDataFrame(week6_status)

display(
    week6_status_df.orderBy("domain", "rule_id")
)

rule_id,domain,severity,status,reason,action
DQ-CHG-001,Chargers,CRITICAL,PASS,Charger Candidate records evaluated and routed.,None
DQ-MNT-001,Maintenance,MAJOR,PASS,Maintenance reference integrity and chronology checks implemented.,None
DQ-SES-001,Sessions,CRITICAL,PASS,Session identity and uniqueness checks implemented.,None
DQ-SES-002,Sessions,CRITICAL,PASS,Station/charger reference and alignment checks implemented.,None
DQ-SES-003,Sessions,MAJOR,PASS,"Timestamp parsing, chronology and reporting-window checks implemented.",None
DQ-SES-004,Sessions,MAJOR,PASS,Lifecycle consistency checks implemented.,None
DQ-SES-005,Sessions,CRITICAL,PASS,Charger overlap and station capacity checks implemented.,None
DQ-SES-006,Sessions,MAJOR,PASS,Measure validity checks implemented.,None
DQ-SES-007,Sessions,MAJOR,BLOCKED,Approved efficiency/tolerance configuration is not available in the Week 5/Week 6 environment.,Obtain approved project configuration before enabling this rule.
DQ-CHG-002,Stations,CRITICAL,PASS,Station Candidate records evaluated and routed.,None


In [0]:
week6_status_df.write.mode("overwrite").saveAsTable(
    "workspace.silver_layer.week6_dq_rule_status"
)

display(
    spark.table(
        "workspace.silver_layer.week6_dq_rule_status"
    )
)

rule_id,domain,severity,status,reason,action
DQ-CHG-002,Stations,CRITICAL,PASS,Station Candidate records evaluated and routed.,None
DQ-CHG-001,Chargers,CRITICAL,PASS,Charger Candidate records evaluated and routed.,None
DQ-SES-001,Sessions,CRITICAL,PASS,Session identity and uniqueness checks implemented.,None
DQ-SES-002,Sessions,CRITICAL,PASS,Station/charger reference and alignment checks implemented.,None
DQ-SES-003,Sessions,MAJOR,PASS,"Timestamp parsing, chronology and reporting-window checks implemented.",None
DQ-SES-004,Sessions,MAJOR,PASS,Lifecycle consistency checks implemented.,None
DQ-SES-005,Sessions,CRITICAL,PASS,Charger overlap and station capacity checks implemented.,None
DQ-SES-006,Sessions,MAJOR,PASS,Measure validity checks implemented.,None
DQ-SES-007,Sessions,MAJOR,BLOCKED,Approved efficiency/tolerance configuration is not available in the Week 5/Week 6 environment.,Obtain approved project configuration before enabling this rule.
DQ-MNT-001,Maintenance,MAJOR,PASS,Maintenance reference integrity and chronology checks implemented.,None


In [0]:
rule_occurrences = spark.table(
    "workspace.silver_layer.dq_session_rule_occurrences"
)

print("Rule occurrences:", rule_occurrences.count())

print(
    "Distinct quarantined physical records:",
    rule_occurrences
        .select("physical_record_id")
        .distinct()
        .count()
)

display(
    rule_occurrences
    .groupBy("rule_id", "rule_name", "severity")
    .count()
    .orderBy("rule_id")
)


Rule occurrences: 23154
Distinct quarantined physical records: 13098


rule_id,rule_name,severity,count
DQ-SES-001,Session identity,CRITICAL,400
DQ-SES-002,Station/charger reference and alignment,CRITICAL,900
DQ-SES-003,Chronology/window,MAJOR,10004
DQ-SES-004,Lifecycle,MAJOR,202
DQ-SES-005,Occupancy/capacity,CRITICAL,1593
DQ-SES-006,Measures/ranges,MAJOR,10055


In [0]:
from pyspark.sql import Row

# ============================================================
# WEEK 6 — FINAL ACCEPTANCE RECONCILIATION
# ============================================================

checks = [
    ("Stations", 
     "workspace.silver_layer.silver_stations_candidate",
     "workspace.silver_layer.trusted_stations",
     "workspace.silver_layer.quarantine_stations"),

    ("Chargers",
     "workspace.silver_layer.silver_chargers_candidate",
     "workspace.silver_layer.trusted_chargers",
     "workspace.silver_layer.quarantine_chargers"),

    ("Sessions",
     "workspace.silver_layer.silver_sessions_candidate",
     "workspace.silver_layer.trusted_sessions",
     "workspace.silver_layer.quarantine_sessions"),

    ("Maintenance",
     "workspace.silver_layer.silver_maintenance_candidate",
     "workspace.silver_layer.trusted_maintenance",
     "workspace.silver_layer.quarantine_maintenance")
]

final_results = []

for domain, candidate_table, trusted_table, quarantine_table in checks:

    candidate_ids = (
        spark.table(candidate_table)
        .select("physical_record_id")
        .distinct()
    )

    trusted_ids = (
        spark.table(trusted_table)
        .select("physical_record_id")
        .distinct()
    )

    quarantine_ids = (
        spark.table(quarantine_table)
        .select("physical_record_id")
        .distinct()
    )

    candidate_count = candidate_ids.count()
    trusted_count = trusted_ids.count()
    quarantine_count = quarantine_ids.count()

    overlap = (
        trusted_ids
        .join(quarantine_ids, "physical_record_id", "inner")
        .count()
    )

    variance = candidate_count - trusted_count - quarantine_count

    final_results.append(
        Row(
            domain=domain,
            candidate=candidate_count,
            trusted=trusted_count,
            quarantine=quarantine_count,
            overlap=overlap,
            variance=variance,
            acceptance=(
                "PASS"
                if overlap == 0 and variance == 0
                else "FAIL"
            )
        )
    )

final_df = spark.createDataFrame(final_results)

display(final_df)

domain,candidate,trusted,quarantine,overlap,variance,acceptance
Stations,180,178,2,0,0,PASS
Chargers,1200,1177,23,0,0,PASS
Sessions,300000,286902,13098,0,0,PASS
Maintenance,18000,17849,151,0,0,PASS


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.silver_layer
    """)
)

database,tableName,isTemporary
silver_layer,dq_session_rule_occurrences,false
silver_layer,quarantine_chargers,false
silver_layer,quarantine_maintenance,false
silver_layer,quarantine_sessions,false
silver_layer,quarantine_stations,false
silver_layer,silver_chargers_candidate,false
silver_layer,silver_maintenance_candidate,false
silver_layer,silver_sessions_candidate,false
silver_layer,silver_stations_candidate,false
silver_layer,trusted_chargers,false


In [0]:
summary = [
    Row(
        item="Station DQ",
        status="PASS",
        evidence="178 Trusted / 2 Quarantine / 0 overlap / 0 variance"
    ),
    Row(
        item="Charger DQ",
        status="PASS",
        evidence="1177 Trusted / 23 Quarantine / 0 overlap / 0 variance"
    ),
    Row(
        item="Session DQ",
        status="PARTIAL",
        evidence="286902 Trusted / 13098 Quarantine; SES-007 blocked"
    ),
    Row(
        item="Maintenance DQ",
        status="PASS",
        evidence="17849 Trusted / 151 Quarantine / 0 overlap / 0 variance"
    ),
    Row(
        item="Session Rule Occurrences",
        status="PASS",
        evidence="23154 rule occurrences / 13098 distinct physical records"
    ),
    Row(
        item="DQ-SES-007",
        status="BLOCKED",
        evidence="Approved efficiency/tolerance configuration unavailable"
    ),
    Row(
        item="DQ-EVT-001",
        status="BLOCKED",
        evidence="Events Bronze/Candidate source unavailable"
    )
]

submission_summary = spark.createDataFrame(summary)

display(submission_summary)

item,status,evidence
Station DQ,PASS,178 Trusted / 2 Quarantine / 0 overlap / 0 variance
Charger DQ,PASS,1177 Trusted / 23 Quarantine / 0 overlap / 0 variance
Session DQ,PARTIAL,286902 Trusted / 13098 Quarantine; SES-007 blocked
Maintenance DQ,PASS,17849 Trusted / 151 Quarantine / 0 overlap / 0 variance
Session Rule Occurrences,PASS,23154 rule occurrences / 13098 distinct physical records
DQ-SES-007,BLOCKED,Approved efficiency/tolerance configuration unavailable
DQ-EVT-001,BLOCKED,Events Bronze/Candidate source unavailable
